> **Cópia pública saneada.** Os dados de entrada não acompanham este repositório. Leia `docs/reprodutibilidade.md` e `docs/privacidade_e_dados.md` antes da execução. Notebooks de coleta dependem de rede; notebooks de tratamento escrevem somente em `data/`, que é ignorada pelo Git.

# Limpeza e tratamento das bases do BNDES

Este notebook organiza o diagnóstico, a limpeza e o tratamento inicial dos datasets baixados do Portal de Dados Abertos do BNDES.

O objetivo desta etapa ainda não é calcular indicadores finais sobre financiamento ambiental ou climático, mas preparar as bases para análises posteriores.

Serão verificados e tratados, para cada dataset:

1. número de linhas;
2. número de colunas;
3. nomes das variáveis;
4. tipos de dados;
5. cobertura temporal;
6. tamanho dos arquivos;
7. variáveis compartilhadas entre bases;
8. necessidades de conversão, padronização e limpeza antes da análise.

Essa etapa é importante porque as bases têm tamanhos, formatos e finalidades diferentes.


## Observação sobre execução do notebook

Este notebook contém alguns diagnósticos que percorrem bases grandes, principalmente `desembolsos_mensais` e `operacoes_indiretas_automaticas`.

Para evitar demora desnecessária, use `Run All` apenas quando for realmente preciso recalcular todos os diagnósticos. Para continuar o tratamento das bases, é suficiente executar a seção de retomada rápida ao final do notebook e seguir a partir dela.

In [ ]:
from pathlib import Path
from datetime import datetime

import pandas as pd

## 1. Localização das pastas do projeto

Neste bloco, definimos os caminhos principais do projeto.

A análise descritiva inicial usará os arquivos CSV brutos salvos em `data/raw/csv` e o diagnóstico de leitura gerado no notebook de armazenamento.

In [ ]:
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

RAW_CSV_DIR = PROJECT_ROOT / "data" / "raw" / "csv"
INTERIM_DIR = PROJECT_ROOT / "data" / "interim"
OUTPUT_TABLES_DIR = PROJECT_ROOT / "results" / "tables"

DIAGNOSTICO_CSV_PATH = OUTPUT_TABLES_DIR / "diagnostico_leitura_csv.xlsx"

PROJECT_ROOT

## 2. Carregamento do diagnóstico de leitura dos CSVs

Neste bloco, carregamos o diagnóstico criado no notebook de armazenamento.

Esse arquivo informa qual encoding funcionou para cada CSV e quantas colunas foram identificadas na leitura inicial. Assim, evitamos repetir testes de leitura e usamos a configuração correta para cada base.

In [ ]:
diagnostico_csv = pd.read_excel(DIAGNOSTICO_CSV_PATH)

diagnostico_csv

## 3. Mapa de encoding dos arquivos

Neste bloco, transformamos o diagnóstico de leitura em um dicionário Python.

Esse dicionário associa cada arquivo CSV ao encoding correto identificado anteriormente. Ele será usado para ler cada base com a configuração adequada.

In [ ]:
mapa_encoding_csv = dict(
    zip(
        diagnostico_csv["nome_arquivo"],
        diagnostico_csv["encoding"]
    )
)

mapa_encoding_csv

## 4. Listagem dos arquivos CSV disponíveis

Neste bloco, listamos os arquivos CSV que estão salvos na pasta `data/raw/csv`.

Essa etapa confirma quais bases estão disponíveis localmente para análise descritiva e permite comparar os arquivos existentes com o diagnóstico de leitura.

In [ ]:
arquivos_csv = sorted(RAW_CSV_DIR.glob("*.csv"))

arquivos_csv

## 5. Tamanho dos arquivos CSV

Neste bloco, criamos uma tabela com o nome e o tamanho de cada arquivo CSV.

Essa informação é importante porque algumas bases são muito grandes. Antes de carregar os dados completos na memória, precisamos saber quais arquivos exigem leitura mais cuidadosa.

In [ ]:
inventario_csv = pd.DataFrame([
    {
        "nome_arquivo": arquivo.name,
        "caminho": arquivo,
        "tamanho_mb": arquivo.stat().st_size / (1024 * 1024)
    }
    for arquivo in arquivos_csv
])

inventario_csv

## 6. Contagem de linhas dos arquivos CSV

Neste bloco, contamos o número de linhas de cada arquivo CSV sem carregar a base inteira na memória.

Essa contagem permite saber o tamanho real de cada dataset em número de registros. Como alguns arquivos são muito grandes, usamos uma leitura linha a linha, mais segura para diagnóstico inicial.

In [ ]:
def contar_linhas_csv(caminho):
    with open(caminho, "rb") as arquivo:
        return sum(1 for _ in arquivo) - 1


inventario_csv["numero_linhas"] = inventario_csv["caminho"].apply(
    contar_linhas_csv
)

inventario_csv[
    ["nome_arquivo", "tamanho_mb", "numero_linhas"]
]

## 7. Inclusão do número de colunas no inventário

Neste bloco, juntamos o inventário dos arquivos locais com o diagnóstico de leitura dos CSVs.

Com isso, teremos em uma única tabela o tamanho do arquivo, o número de linhas, o número de colunas e o encoding correto de cada base.

In [ ]:
inventario_csv = inventario_csv.merge(
    diagnostico_csv[
        ["nome_arquivo", "encoding", "quantidade_colunas"]
    ],
    on="nome_arquivo",
    how="left"
)

inventario_csv[
    [
        "nome_arquivo",
        "tamanho_mb",
        "numero_linhas",
        "quantidade_colunas",
        "encoding",
    ]
]

## 8. Padronização dos nomes dos datasets

Neste bloco, criamos nomes simples para os datasets, que serão usados nas tabelas e diagnósticos do notebook.

A ideia é evitar nomes longos na visualização dos resultados, mantendo os identificadores mais legíveis para a análise.

In [ ]:
mapa_nome_dataset = {
    "desembolsos_mensais__desembolsos_mensais.csv": "desembolsos_mensais",
    "desembolsos_mensais__mapeamento_de_bndes_para_cnae.csv": "mapeamento_bndes_cnae",
    "fontes_de_recursos__fontes_de_recursos_do_bndes.csv": "fontes_recursos",
    "operacoes_de_financiamento__operacoes_indiretas_automaticas.csv": "operacoes_indiretas_automaticas",
    "operacoes_de_financiamento__operacoes_nao_automaticas.csv": "operacoes_nao_automaticas",
    "politicas_operacionais__politicas_operacionais.csv": "politicas_operacionais",
}

inventario_csv["arquivo_original"] = inventario_csv["nome_arquivo"]
inventario_csv["dataset"] = inventario_csv["arquivo_original"].map(mapa_nome_dataset)
inventario_csv["nome_arquivo"] = inventario_csv["dataset"]

inventario_csv[
    [
        "dataset",
        "numero_linhas",
        "quantidade_colunas",
        "tamanho_mb",
        "encoding",
    ]
]

## 9. Salvamento do inventário descritivo inicial

Neste bloco, salvamos o inventário inicial dos datasets em um arquivo Excel.

Esse arquivo registra, para cada base, o nome simplificado do dataset, o arquivo original, o número de linhas, o número de colunas, o tamanho aproximado e o encoding.

Ele funciona como uma primeira documentação descritiva das bases disponíveis.

In [ ]:
arquivo_inventario_descritivo = (
    OUTPUT_TABLES_DIR / "inventario_descritivo_datasets.xlsx"
)

inventario_csv[
    [
        "dataset",
        "nome_arquivo",
        "arquivo_original",
        "numero_linhas",
        "quantidade_colunas",
        "tamanho_mb",
        "encoding",
        "caminho",
    ]
].to_excel(
    arquivo_inventario_descritivo,
    index=False,
    sheet_name="Inventario"
)

arquivo_inventario_descritivo

## 10. Leitura das colunas de cada dataset

Neste bloco, lemos apenas as primeiras linhas de cada CSV para identificar suas colunas.

A tabela final mostra somente o nome do dataset, a ordem da coluna e o nome da variável. Isso facilita a leitura visual e evita repetição de informações técnicas.

In [ ]:
colunas_datasets = []

for _, linha in inventario_csv.iterrows():
    amostra = pd.read_csv(
        linha["caminho"],
        sep=";",
        encoding=linha["encoding"],
        nrows=5,
        low_memory=False
    )

    for ordem, coluna in enumerate(amostra.columns, start=1):
        colunas_datasets.append({
            "dataset": linha["dataset"],
            "ordem_coluna": ordem,
            "coluna": coluna,
        })

colunas_datasets = pd.DataFrame(colunas_datasets)

colunas_datasets

## 11. Quantidade de colunas por dataset

Neste bloco, calculamos quantas colunas existem em cada dataset.

Essa tabela resume a estrutura de cada base e permite identificar quais datasets são mais amplos em termos de variáveis disponíveis.

In [ ]:
resumo_colunas_dataset = (
    colunas_datasets
    .groupby("dataset")
    .agg(
        quantidade_colunas=("coluna", "count")
    )
    .reset_index()
    .sort_values("quantidade_colunas", ascending=False)
)

resumo_colunas_dataset

## 12. Lista de variáveis por dataset

Neste bloco, organizamos a lista de variáveis disponíveis em cada dataset.

A tabela mostra, para cada base, a sequência das colunas e o nome de cada variável. Essa visualização ajuda a identificar quais informações existem em cada dataset e quais variáveis poderão ser usadas nas próximas etapas da análise.

In [ ]:
lista_variaveis_dataset = (
    colunas_datasets
    .sort_values(["dataset", "ordem_coluna"])
    .reset_index(drop=True)
)

lista_variaveis_dataset

## 13. Variáveis comuns entre datasets

Neste bloco, identificamos quais variáveis aparecem em mais de um dataset.

Essa etapa ajuda a reconhecer campos compartilhados entre bases, como setor, município, UF, produto, instrumento financeiro ou forma de apoio. Essas variáveis podem ser importantes para cruzamentos, comparações e padronização posterior.

In [ ]:
variaveis_comuns = (
    colunas_datasets
    .groupby("coluna")
    .agg(
        quantidade_datasets=("dataset", "nunique"),
        datasets=("dataset", lambda x: ", ".join(sorted(x.unique())))
    )
    .reset_index()
    .query("quantidade_datasets > 1")
    .sort_values(
        ["quantidade_datasets", "coluna"],
        ascending=[False, True]
    )
)

variaveis_comuns

## 14. Classificação preliminar das variáveis comuns

Neste bloco, classificamos as variáveis comuns entre datasets segundo sua função analítica preliminar.

Essa classificação ajuda a identificar quais variáveis podem ser usadas para recortes territoriais, setoriais, financeiros, temporais e institucionais.

In [ ]:
mapa_tipo_variavel = {
    "uf": "territorial",
    "municipio": "territorial",
    "municipio_codigo": "territorial",

    "setor_bndes": "setorial",
    "subsetor_bndes": "setorial",
    "setor_cnae": "setorial",
    "subsetor_cnae_agrupado": "setorial",
    "subsetor_cnae_codigo": "setorial",
    "subsetor_cnae_nome": "setorial",

    "produto": "instrumento_financeiro",
    "instrumento_financeiro": "instrumento_financeiro",
    "fonte_de_recurso_desembolsos": "instrumento_financeiro",
    "forma_de_apoio": "instrumento_financeiro",
    "modalidade_de_apoio": "instrumento_financeiro",
    "custo_financeiro": "instrumento_financeiro",
    "juros": "instrumento_financeiro",

    "valor_desembolsado_reais": "valor",
    "valor_contratado_reais": "valor",

    "data_da_contratacao": "temporal",

    "cliente": "cliente",
    "porte_do_cliente": "cliente",
    "natureza_do_cliente": "cliente",
    "area_operacional": "institucional",
    "instituicao_financeira_credenciada": "institucional",
    "inovacao": "atributo",
}

variaveis_comuns["tipo_analitico"] = (
    variaveis_comuns["coluna"]
    .map(mapa_tipo_variavel)
    .fillna("outros")
)

variaveis_comuns[
    [
        "tipo_analitico",
        "coluna",
        "quantidade_datasets",
        "datasets",
    ]
].sort_values(["tipo_analitico", "coluna"])

## 15. Resumo das variáveis comuns por tipo analítico

Neste bloco, calculamos quantas variáveis comuns existem em cada grupo analítico.

Esse resumo ajuda a visualizar quais dimensões estão mais bem representadas nas bases e quais serão mais relevantes para a análise descritiva posterior.

In [ ]:
resumo_tipo_variavel = (
    variaveis_comuns
    .groupby("tipo_analitico")
    .agg(
        quantidade_variaveis=("coluna", "count"),
        variaveis=("coluna", lambda x: ", ".join(sorted(x)))
    )
    .reset_index()
    .sort_values("quantidade_variaveis", ascending=False)
)

resumo_tipo_variavel

## 16. Variáveis exclusivas de cada dataset

Neste bloco, identificamos as variáveis que aparecem em apenas um dataset.

Essas variáveis são importantes porque representam informações específicas de cada base. Por exemplo, algumas variáveis podem existir apenas nas operações não automáticas, enquanto outras aparecem apenas em políticas operacionais ou fontes de recursos.

In [ ]:
variaveis_exclusivas = (
    colunas_datasets
    .groupby("coluna")
    .agg(
        quantidade_datasets=("dataset", "nunique"),
        dataset=("dataset", lambda x: sorted(x.unique())[0])
    )
    .reset_index()
    .query("quantidade_datasets == 1")
    .sort_values(["dataset", "coluna"])
)

variaveis_exclusivas

## 17. Quantidade de variáveis exclusivas por dataset

Neste bloco, calculamos quantas variáveis aparecem exclusivamente em cada dataset.

Esse resumo ajuda a identificar quais bases possuem informações mais específicas e quais variáveis podem exigir tratamento próprio nas próximas etapas.

In [ ]:
resumo_variaveis_exclusivas = (
    variaveis_exclusivas
    .groupby("dataset")
    .agg(
        quantidade_variaveis_exclusivas=("coluna", "count"),
        variaveis_exclusivas=("coluna", lambda x: ", ".join(sorted(x)))
    )
    .reset_index()
    .sort_values("quantidade_variaveis_exclusivas", ascending=False)
)

resumo_variaveis_exclusivas

## 18. Amostra inicial de cada dataset

Neste bloco, lemos uma pequena amostra de cada dataset.

O objetivo é observar o conteúdo real das bases, não apenas os nomes das colunas. Essa inspeção ajuda a identificar problemas de tratamento, como textos em maiúsculas, valores monetários, datas, códigos, campos vazios ou categorias que precisam ser padronizadas.

In [ ]:
amostras_datasets = {}

for _, linha in inventario_csv.iterrows():
    amostras_datasets[linha["dataset"]] = pd.read_csv(
        linha["caminho"],
        sep=";",
        encoding=linha["encoding"],
        nrows=5,
        low_memory=False
    )

list(amostras_datasets.keys())

## 19. Diagnóstico de tipos e preenchimento das variáveis

Neste bloco, avaliamos os tipos de dados identificados pelo Python em cada dataset.

Para cada variável, registramos o tipo inferido, a quantidade de valores não nulos, a quantidade de valores ausentes e o percentual de preenchimento. Essa etapa é equivalente a uma versão tabular do `df.info()`, mais adequada para comparar datasets.

In [ ]:
diagnostico_variaveis = []

for dataset, amostra in amostras_datasets.items():
    total_linhas_amostra = len(amostra)

    for coluna in amostra.columns:
        nao_nulos = amostra[coluna].notna().sum()
        ausentes = amostra[coluna].isna().sum()

        diagnostico_variaveis.append({
            "dataset": dataset,
            "coluna": coluna,
            "tipo_inferido_amostra": str(amostra[coluna].dtype),
            "nao_nulos_amostra": nao_nulos,
            "ausentes_amostra": ausentes,
            "percentual_preenchido_amostra": nao_nulos / total_linhas_amostra
        })

diagnostico_variaveis = pd.DataFrame(diagnostico_variaveis)

diagnostico_variaveis

## 20. Plano de correção dos tipos de variáveis

Neste bloco, registramos quais variáveis precisarão ter o tipo corrigido nas próximas etapas.

Essa distinção é importante porque nem toda variável com aparência numérica deve ser tratada como número. Códigos de município, CNPJ, CPF/CNPJ e número de contrato são identificadores. Por outro lado, valores monetários, juros e percentuais precisam ser convertidos para formato numérico para permitir somas, médias e agregações.

In [ ]:
plano_conversao_tipos = pd.DataFrame([
    {"dataset": "desembolsos_mensais", "coluna": "municipio_codigo", "tipo_desejado": "texto", "motivo": "Código IBGE é identificador, não variável quantitativa."},
    {"dataset": "desembolsos_mensais", "coluna": "desembolsos_reais", "tipo_desejado": "número decimal", "motivo": "Valor monetário com vírgula decimal."},
    {"dataset": "desembolsos_mensais", "coluna": "ano_mes", "tipo_desejado": "data mensal", "motivo": "Será criada a partir de ano e mês para análise temporal."},

    {"dataset": "fontes_recursos", "coluna": "datas", "tipo_desejado": "data", "motivo": "Representa a data de referência da informação."},
    {"dataset": "fontes_recursos", "coluna": "patrimonio_liquido", "tipo_desejado": "número decimal", "motivo": "Valor financeiro."},
    {"dataset": "fontes_recursos", "coluna": "tesouro_nacional", "tipo_desejado": "número decimal", "motivo": "Valor financeiro."},
    {"dataset": "fontes_recursos", "coluna": "fat", "tipo_desejado": "número decimal", "motivo": "Valor financeiro."},
    {"dataset": "fontes_recursos", "coluna": "captacoes_internas", "tipo_desejado": "número decimal", "motivo": "Valor financeiro."},
    {"dataset": "fontes_recursos", "coluna": "fundos", "tipo_desejado": "número decimal", "motivo": "Valor financeiro."},
    {"dataset": "fontes_recursos", "coluna": "operacoes_compromissadas", "tipo_desejado": "número decimal", "motivo": "Valor financeiro."},
    {"dataset": "fontes_recursos", "coluna": "captacoes_externas", "tipo_desejado": "número decimal", "motivo": "Valor financeiro."},
    {"dataset": "fontes_recursos", "coluna": "total_financeiro", "tipo_desejado": "número decimal", "motivo": "Valor financeiro."},
    {"dataset": "fontes_recursos", "coluna": "outros_passivos", "tipo_desejado": "número decimal", "motivo": "Valor financeiro."},
    {"dataset": "fontes_recursos", "coluna": "passivo_total", "tipo_desejado": "número decimal", "motivo": "Valor financeiro."},

    {"dataset": "operacoes_indiretas_automaticas", "coluna": "cpf_cnpj", "tipo_desejado": "texto", "motivo": "Identificador mascarado do cliente."},
    {"dataset": "operacoes_indiretas_automaticas", "coluna": "municipio_codigo", "tipo_desejado": "texto", "motivo": "Código IBGE é identificador."},
    {"dataset": "operacoes_indiretas_automaticas", "coluna": "data_da_contratacao", "tipo_desejado": "data", "motivo": "Data da operação."},
    {"dataset": "operacoes_indiretas_automaticas", "coluna": "valor_da_operacao_em_reais", "tipo_desejado": "número decimal", "motivo": "Valor monetário."},
    {"dataset": "operacoes_indiretas_automaticas", "coluna": "valor_desembolsado_reais", "tipo_desejado": "número decimal", "motivo": "Valor monetário com vírgula decimal."},
    {"dataset": "operacoes_indiretas_automaticas", "coluna": "juros", "tipo_desejado": "número decimal", "motivo": "Taxa de juros."},
    {"dataset": "operacoes_indiretas_automaticas", "coluna": "subsetor_cnae_codigo", "tipo_desejado": "texto", "motivo": "Código CNAE é identificador."},
    {"dataset": "operacoes_indiretas_automaticas", "coluna": "cnpj_do_agente_financeiro", "tipo_desejado": "texto", "motivo": "Identificador institucional."},

    {"dataset": "operacoes_nao_automaticas", "coluna": "cnpj", "tipo_desejado": "texto", "motivo": "Identificador do cliente."},
    {"dataset": "operacoes_nao_automaticas", "coluna": "municipio_codigo", "tipo_desejado": "texto", "motivo": "Código IBGE é identificador."},
    {"dataset": "operacoes_nao_automaticas", "coluna": "numero_do_contrato", "tipo_desejado": "texto", "motivo": "Número de contrato é identificador."},
    {"dataset": "operacoes_nao_automaticas", "coluna": "data_da_contratacao", "tipo_desejado": "data", "motivo": "Data da operação."},
    {"dataset": "operacoes_nao_automaticas", "coluna": "valor_contratado_reais", "tipo_desejado": "número decimal", "motivo": "Valor monetário com vírgula decimal."},
    {"dataset": "operacoes_nao_automaticas", "coluna": "valor_desembolsado_reais", "tipo_desejado": "número decimal", "motivo": "Valor monetário com vírgula decimal."},
    {"dataset": "operacoes_nao_automaticas", "coluna": "juros", "tipo_desejado": "número decimal", "motivo": "Taxa de juros."},
    {"dataset": "operacoes_nao_automaticas", "coluna": "subsetor_cnae_codigo", "tipo_desejado": "texto", "motivo": "Código CNAE é identificador."},
    {"dataset": "operacoes_nao_automaticas", "coluna": "cnpj_da_instituicao_financeira_credenciada", "tipo_desejado": "texto", "motivo": "Identificador institucional."},

    {"dataset": "mapeamento_bndes_cnae", "coluna": "codigo_cnae_ibge", "tipo_desejado": "texto", "motivo": "Código ou intervalo CNAE, como 'A01 a A03'."},

    {"dataset": "politicas_operacionais", "coluna": "taxa_bndes_a_a", "tipo_desejado": "número percentual", "motivo": "Percentual informado com símbolo %."},
    {"dataset": "politicas_operacionais", "coluna": "participacao_maxima_bndes", "tipo_desejado": "número percentual", "motivo": "Percentual informado com símbolo %."},
    {"dataset": "politicas_operacionais", "coluna": "prazo_total_maximo", "tipo_desejado": "número inteiro", "motivo": "Prazo informado como texto, por exemplo '20 anos'."},
])

plano_conversao_tipos = plano_conversao_tipos.merge(
    diagnostico_variaveis[["dataset", "coluna", "tipo_inferido_amostra"]],
    on=["dataset", "coluna"],
    how="left"
)

plano_conversao_tipos = plano_conversao_tipos[
    ["dataset", "coluna", "tipo_inferido_amostra", "tipo_desejado", "motivo"]
].sort_values(["dataset", "coluna"])

plano_conversao_tipos

## 21. Teste de conversão dos tipos nas amostras

Neste bloco, aplicamos as correções de tipo apenas nas amostras dos datasets.

O objetivo é verificar se as regras de conversão funcionam antes de aplicá-las nas bases completas. Primeiro tratamos valores monetários e percentuais, depois convertemos datas e preservamos identificadores como texto.

In [ ]:
def converter_decimal_brasileiro(serie):
    return (
        serie
        .astype("string")
        .str.strip()
        .str.replace(".", "", regex=False)
        .str.replace(",", ".", regex=False)
        .replace({"": pd.NA, "nan": pd.NA, "None": pd.NA})
        .pipe(pd.to_numeric, errors="coerce")
    )


def converter_percentual(serie):
    return (
        serie
        .astype("string")
        .str.strip()
        .str.replace("%", "", regex=False)
        .str.replace(",", ".", regex=False)
        .replace({"": pd.NA, "nan": pd.NA, "None": pd.NA})
        .pipe(pd.to_numeric, errors="coerce")
    )


amostras_tratadas = {
    dataset: dados.copy()
    for dataset, dados in amostras_datasets.items()
}

# Desembolsos mensais
amostras_tratadas["desembolsos_mensais"]["municipio_codigo"] = (
    amostras_tratadas["desembolsos_mensais"]["municipio_codigo"].astype("string")
)

amostras_tratadas["desembolsos_mensais"]["desembolsos_reais"] = converter_decimal_brasileiro(
    amostras_tratadas["desembolsos_mensais"]["desembolsos_reais"]
)

amostras_tratadas["desembolsos_mensais"]["ano_mes"] = pd.to_datetime(
    amostras_tratadas["desembolsos_mensais"]["ano"].astype(str)
    + "-"
    + amostras_tratadas["desembolsos_mensais"]["mes"].astype(str).str.zfill(2)
    + "-01",
    errors="coerce"
)

# Fontes de recursos
amostras_tratadas["fontes_recursos"]["datas"] = pd.to_datetime(
    amostras_tratadas["fontes_recursos"]["datas"],
    errors="coerce"
)

colunas_monetarias_fontes = [
    "patrimonio_liquido",
    "tesouro_nacional",
    "fat",
    "captacoes_internas",
    "fundos",
    "operacoes_compromissadas",
    "captacoes_externas",
    "total_financeiro",
    "outros_passivos",
    "passivo_total",
]

for coluna in colunas_monetarias_fontes:
    amostras_tratadas["fontes_recursos"][coluna] = converter_decimal_brasileiro(
        amostras_tratadas["fontes_recursos"][coluna]
    )

# Operações indiretas automáticas
amostras_tratadas["operacoes_indiretas_automaticas"]["cpf_cnpj"] = (
    amostras_tratadas["operacoes_indiretas_automaticas"]["cpf_cnpj"].astype("string")
)

amostras_tratadas["operacoes_indiretas_automaticas"]["municipio_codigo"] = (
    amostras_tratadas["operacoes_indiretas_automaticas"]["municipio_codigo"].astype("string")
)

amostras_tratadas["operacoes_indiretas_automaticas"]["data_da_contratacao"] = pd.to_datetime(
    amostras_tratadas["operacoes_indiretas_automaticas"]["data_da_contratacao"],
    errors="coerce"
)

for coluna in ["valor_da_operacao_em_reais", "valor_desembolsado_reais", "juros"]:
    amostras_tratadas["operacoes_indiretas_automaticas"][coluna] = converter_decimal_brasileiro(
        amostras_tratadas["operacoes_indiretas_automaticas"][coluna]
    )

# Operações não automáticas
for coluna in ["cnpj", "municipio_codigo", "numero_do_contrato"]:
    amostras_tratadas["operacoes_nao_automaticas"][coluna] = (
        amostras_tratadas["operacoes_nao_automaticas"][coluna].astype("string")
    )

amostras_tratadas["operacoes_nao_automaticas"]["data_da_contratacao"] = pd.to_datetime(
    amostras_tratadas["operacoes_nao_automaticas"]["data_da_contratacao"],
    errors="coerce"
)

for coluna in ["valor_contratado_reais", "valor_desembolsado_reais", "juros"]:
    amostras_tratadas["operacoes_nao_automaticas"][coluna] = converter_decimal_brasileiro(
        amostras_tratadas["operacoes_nao_automaticas"][coluna]
    )

# Políticas operacionais
for coluna in ["taxa_bndes_a_a", "participacao_maxima_bndes"]:
    amostras_tratadas["politicas_operacionais"][coluna] = converter_percentual(
        amostras_tratadas["politicas_operacionais"][coluna]
    )

amostras_tratadas["politicas_operacionais"]["prazo_total_maximo"] = (
    amostras_tratadas["politicas_operacionais"]["prazo_total_maximo"]
    .astype("string")
    .str.extract(r"(\d+)", expand=False)
    .pipe(pd.to_numeric, errors="coerce")
)

amostras_tratadas["operacoes_nao_automaticas"].dtypes

## 22. Comparação dos tipos antes e depois da conversão

Neste bloco, comparamos os tipos originais das amostras com os tipos obtidos após o tratamento.

As datas são exibidas no formato brasileiro, sem horário. Os valores numéricos também são exibidos com separador brasileiro: ponto para milhar e vírgula para decimal. Essa formatação é apenas visual; internamente, os valores continuam numéricos para permitir cálculos.

In [ ]:
def formatar_numero_brasileiro(valor, casas_decimais=2, percentual=False):
    if pd.isna(valor):
        return pd.NA

    texto = f"{float(valor):,.{casas_decimais}f}"
    texto = texto.replace(",", "X").replace(".", ",").replace("X", ".")

    if percentual:
        texto = texto + "%"

    return texto


def formatar_exemplo_brasileiro(serie, tipo_desejado):
    exemplos = serie.dropna().head(3)
    tipo_desejado = str(tipo_desejado).lower()

    if pd.api.types.is_datetime64_any_dtype(serie):
        return exemplos.dt.strftime("%d/%m/%Y").tolist()

    if pd.api.types.is_numeric_dtype(serie):
        casas_decimais = 0 if "inteiro" in tipo_desejado else 2
        percentual = "percentual" in tipo_desejado

        return [
            formatar_numero_brasileiro(
                valor,
                casas_decimais=casas_decimais,
                percentual=percentual
            )
            for valor in exemplos
        ]

    return exemplos.tolist()


diagnostico_pos_conversao = []

for _, linha in plano_conversao_tipos.iterrows():
    dataset = linha["dataset"]
    coluna = linha["coluna"]

    if dataset not in amostras_tratadas:
        continue

    if coluna not in amostras_tratadas[dataset].columns:
        continue

    if coluna in amostras_datasets[dataset].columns:
        tipo_antes = str(amostras_datasets[dataset][coluna].dtype)
    else:
        tipo_antes = "variavel_criada"

    diagnostico_pos_conversao.append({
        "dataset": dataset,
        "coluna": coluna,
        "tipo_antes": tipo_antes,
        "tipo_depois": str(amostras_tratadas[dataset][coluna].dtype),
        "tipo_desejado": linha["tipo_desejado"],
        "valores_ausentes_depois": amostras_tratadas[dataset][coluna].isna().sum(),
        "exemplo_depois": formatar_exemplo_brasileiro(
            amostras_tratadas[dataset][coluna],
            linha["tipo_desejado"]
        )
    })

diagnostico_pos_conversao = pd.DataFrame(diagnostico_pos_conversao)

diagnostico_pos_conversao

## 23. Resumo da validação das conversões

Neste bloco, criamos uma coluna de status para avaliar se cada variável ficou compatível com o tipo desejado.

Esse resumo ajuda a separar as conversões aprovadas daquelas que ainda precisam de revisão antes de aplicar o tratamento nas bases completas.

In [ ]:
def avaliar_conversao(tipo_depois, tipo_desejado):
    tipo_depois = str(tipo_depois).lower()
    tipo_desejado = str(tipo_desejado).lower()

    if "texto" in tipo_desejado and (
        "string" in tipo_depois
        or "str" in tipo_depois
        or "object" in tipo_depois
    ):
        return "ok"

    if "data" in tipo_desejado and "datetime" in tipo_depois:
        return "ok"

    if "número" in tipo_desejado and (
        "float" in tipo_depois
        or "int" in tipo_depois
        or "double" in tipo_depois
    ):
        return "ok"

    return "revisar"


diagnostico_pos_conversao["status_conversao"] = diagnostico_pos_conversao.apply(
    lambda linha: avaliar_conversao(
        linha["tipo_depois"],
        linha["tipo_desejado"]
    ),
    axis=1
)

resumo_status_conversao = (
    diagnostico_pos_conversao
    .groupby("status_conversao")
    .size()
    .reset_index(name="quantidade_variaveis")
)

resumo_status_conversao

## 24. Tabela final das variáveis com tipo corrigido

Neste bloco, organizamos a tabela final das variáveis que precisarão de correção de tipo nas bases completas.

Essa tabela documenta quais variáveis serão tratadas, qual era o tipo original, qual será o tipo analítico e por que essa conversão é necessária.

In [ ]:
tabela_tratamento_tipos = (
    diagnostico_pos_conversao
    .merge(
        plano_conversao_tipos[["dataset", "coluna", "motivo"]],
        on=["dataset", "coluna"],
        how="left"
    )
    .sort_values(["dataset", "coluna"])
    .reset_index(drop=True)
)

tabela_tratamento_tipos

## 25. Exportação do diagnóstico de tipos

Neste bloco, exportamos a tabela de tratamento dos tipos para Excel.

Esse arquivo documenta as variáveis que precisarão de conversão nas bases completas e serve como registro metodológico do processo de limpeza dos dados.

In [ ]:
arquivo_tratamento_tipos = OUTPUT_TABLES_DIR / "diagnostico_tratamento_tipos.xlsx"

with pd.ExcelWriter(arquivo_tratamento_tipos, engine="openpyxl") as writer:
    plano_conversao_tipos.to_excel(
        writer,
        sheet_name="Plano_Conversao",
        index=False
    )

    diagnostico_pos_conversao.to_excel(
        writer,
        sheet_name="Teste_Amostras",
        index=False
    )

    tabela_tratamento_tipos.to_excel(
        writer,
        sheet_name="Tabela_Final",
        index=False
    )

arquivo_tratamento_tipos

## 26. Cobertura temporal dos datasets

Neste bloco, identificamos a cobertura temporal de cada dataset.

Essa etapa permite saber quais bases têm informação temporal, qual é a primeira e a última data observada e se a variável temporal aparece como data completa ou como combinação de ano e mês.

In [ ]:
cobertura_temporal = []

for _, linha in inventario_csv.iterrows():
    dataset = linha["dataset"]
    caminho = linha["caminho"]
    encoding = linha["encoding"]

    colunas = pd.read_csv(
        caminho,
        sep=";",
        encoding=encoding,
        nrows=0
    ).columns.tolist()

    if "data_da_contratacao" in colunas:
        serie_data = pd.read_csv(
            caminho,
            sep=";",
            encoding=encoding,
            usecols=["data_da_contratacao"],
            low_memory=False
        )["data_da_contratacao"]

        serie_data = pd.to_datetime(serie_data, errors="coerce")

        cobertura_temporal.append({
            "dataset": dataset,
            "variavel_temporal": "data_da_contratacao",
            "primeira_data": serie_data.min().strftime("%d/%m/%Y"),
            "ultima_data": serie_data.max().strftime("%d/%m/%Y"),
            "quantidade_observacoes_com_data": serie_data.notna().sum(),
            "quantidade_observacoes_sem_data": serie_data.isna().sum()
        })

    elif {"ano", "mes"}.issubset(colunas):
        dados_tempo = pd.read_csv(
            caminho,
            sep=";",
            encoding=encoding,
            usecols=["ano", "mes"],
            low_memory=False
        )

        serie_data = pd.to_datetime(
            dados_tempo["ano"].astype(str)
            + "-"
            + dados_tempo["mes"].astype(str).str.zfill(2)
            + "-01",
            errors="coerce"
        )

        cobertura_temporal.append({
            "dataset": dataset,
            "variavel_temporal": "ano + mes",
            "primeira_data": serie_data.min().strftime("%d/%m/%Y"),
            "ultima_data": serie_data.max().strftime("%d/%m/%Y"),
            "quantidade_observacoes_com_data": serie_data.notna().sum(),
            "quantidade_observacoes_sem_data": serie_data.isna().sum()
        })

    elif "datas" in colunas:
        serie_data = pd.read_csv(
            caminho,
            sep=";",
            encoding=encoding,
            usecols=["datas"],
            low_memory=False
        )["datas"]

        serie_data = pd.to_datetime(serie_data, errors="coerce")

        cobertura_temporal.append({
            "dataset": dataset,
            "variavel_temporal": "datas",
            "primeira_data": serie_data.min().strftime("%d/%m/%Y"),
            "ultima_data": serie_data.max().strftime("%d/%m/%Y"),
            "quantidade_observacoes_com_data": serie_data.notna().sum(),
            "quantidade_observacoes_sem_data": serie_data.isna().sum()
        })

    else:
        cobertura_temporal.append({
            "dataset": dataset,
            "variavel_temporal": "sem variável temporal identificada",
            "primeira_data": pd.NA,
            "ultima_data": pd.NA,
            "quantidade_observacoes_com_data": pd.NA,
            "quantidade_observacoes_sem_data": pd.NA
        })

cobertura_temporal = pd.DataFrame(cobertura_temporal)

cobertura_temporal

## 27. Síntese da cobertura temporal

Neste bloco, organizamos a cobertura temporal em um texto sintético.

Essa síntese ajuda a registrar, no próprio notebook, quais bases possuem dimensão temporal e qual é o período comum mais adequado para análises integradas.

In [ ]:
bases_com_tempo = cobertura_temporal.dropna(
    subset=["primeira_data", "ultima_data"]
).copy()

primeira_data_comum = bases_com_tempo["primeira_data"].max()
ultima_data_comum = bases_com_tempo["ultima_data"].min()

print("Cobertura temporal identificada:")
for _, linha in cobertura_temporal.iterrows():
    if pd.notna(linha["primeira_data"]):
        print(
            f"- {linha['dataset']}: de {linha['primeira_data']} "
            f"a {linha['ultima_data']} "
            f"({linha['variavel_temporal']})."
        )
    else:
        print(
            f"- {linha['dataset']}: sem variável temporal explícita."
        )

print()
print(
    f"Período comum entre as bases com dimensão temporal: "
    f"{primeira_data_comum} a {ultima_data_comum}."
)

## 28. Diagnóstico de valores ausentes

Neste bloco, calculamos a quantidade e o percentual de valores ausentes por variável em cada dataset.

Essa etapa permite identificar variáveis com baixa cobertura, campos que exigem tratamento específico e possíveis limitações para as análises posteriores.

In [ ]:
diagnostico_ausentes = []

for _, linha in inventario_csv.iterrows():
    dataset = linha["dataset"]
    caminho = linha["caminho"]
    encoding = linha["encoding"]

    dados = pd.read_csv(
        caminho,
        sep=";",
        encoding=encoding,
        low_memory=False
    )

    total_linhas = len(dados)

    for coluna in dados.columns:
        quantidade_ausentes = dados[coluna].isna().sum()
        percentual_ausente = quantidade_ausentes / total_linhas

        diagnostico_ausentes.append({
            "dataset": dataset,
            "coluna": coluna,
            "total_linhas": total_linhas,
            "quantidade_ausentes": quantidade_ausentes,
            "percentual_ausente": percentual_ausente
        })

diagnostico_ausentes = pd.DataFrame(diagnostico_ausentes)

diagnostico_ausentes.sort_values(
    ["percentual_ausente", "quantidade_ausentes"],
    ascending=False
)

## 29. Resumo de valores ausentes por dataset

Neste bloco, agregamos o diagnóstico de valores ausentes por dataset.

O objetivo é identificar quais bases concentram mais variáveis com ausência de informação e qual é a intensidade média dos valores ausentes em cada base.

In [ ]:
resumo_ausentes_dataset = (
    diagnostico_ausentes
    .assign(tem_ausente=lambda df: df["quantidade_ausentes"] > 0)
    .groupby("dataset")
    .agg(
        total_variaveis=("coluna", "count"),
        variaveis_com_ausentes=("tem_ausente", "sum"),
        total_linhas=("total_linhas", "max"),
        maior_percentual_ausente=("percentual_ausente", "max"),
        media_percentual_ausente=("percentual_ausente", "mean")
    )
    .reset_index()
    .sort_values("maior_percentual_ausente", ascending=False)
)

resumo_ausentes_dataset

## 30. Variáveis com valores ausentes

Neste bloco, filtramos apenas as variáveis que possuem ao menos um valor ausente.

Essa visualização ajuda a identificar quais campos exigem decisão de tratamento, como manter ausente, preencher com categoria específica ou apenas documentar a limitação.

In [ ]:
variaveis_com_ausentes = (
    diagnostico_ausentes
    .query("quantidade_ausentes > 0")
    .sort_values(["percentual_ausente", "quantidade_ausentes"], ascending=False)
    .reset_index(drop=True)
)

variaveis_com_ausentes

## 31. Decisão preliminar de tratamento dos valores ausentes

Neste bloco, classificamos as variáveis com valores ausentes segundo a decisão preliminar de tratamento.

A decisão depende da função analítica de cada variável. Alguns ausentes devem ser mantidos e documentados; outros podem ser preenchidos com uma categoria explícita, como "Não informado", quando a variável for categórica.

In [ ]:
decisao_ausentes = variaveis_com_ausentes.copy()

def definir_tratamento_ausente(dataset, coluna):
    if dataset == "politicas_operacionais":
        return "manter_ausente_documentar"

    if dataset == "fontes_recursos" and coluna == "passivo_total":
        return "manter_ausente_documentar"

    if dataset == "desembolsos_mensais" and coluna in ["instrumento_financeiro", "inovacao"]:
        return "preencher_nao_informado"

    if dataset == "operacoes_indiretas_automaticas" and coluna in ["valor_desembolsado_reais", "juros"]:
        return "manter_ausente_documentar"

    return "avaliar"


decisao_ausentes["tratamento_preliminar"] = decisao_ausentes.apply(
    lambda linha: definir_tratamento_ausente(
        linha["dataset"],
        linha["coluna"]
    ),
    axis=1
)

decisao_ausentes

## 32. Diagnóstico de registros duplicados

Neste bloco, verificamos a existência de registros duplicados em cada dataset.

A identificação de duplicados é importante para evitar dupla contagem de operações, desembolsos ou registros auxiliares. Inicialmente, verificamos duplicidade considerando todas as colunas de cada base.

In [ ]:
diagnostico_duplicados = []

for _, linha in inventario_csv.iterrows():
    dataset = linha["dataset"]
    caminho = linha["caminho"]
    encoding = linha["encoding"]

    dados = pd.read_csv(
        caminho,
        sep=";",
        encoding=encoding,
        low_memory=False
    )

    total_linhas = len(dados)
    quantidade_duplicados = dados.duplicated().sum()

    diagnostico_duplicados.append({
        "dataset": dataset,
        "total_linhas": total_linhas,
        "quantidade_duplicados": quantidade_duplicados,
        "percentual_duplicados": quantidade_duplicados / total_linhas
    })

diagnostico_duplicados = pd.DataFrame(diagnostico_duplicados)

diagnostico_duplicados.sort_values(
    "percentual_duplicados",
    ascending=False
)

## 33. Inspeção dos registros duplicados

Neste bloco, separamos exemplos de registros duplicados nas bases em que foram encontrados duplicados exatos.

A inspeção é necessária porque duplicidade aparente nem sempre significa erro de base. Antes de remover registros, é preciso verificar se as linhas repetidas representam duplicatas reais ou alguma característica da forma de divulgação dos dados.

In [ ]:
exemplos_duplicados = {}

for dataset in ["operacoes_indiretas_automaticas", "operacoes_nao_automaticas"]:
    linha_base = inventario_csv.query("dataset == @dataset").iloc[0]

    dados = pd.read_csv(
        linha_base["caminho"],
        sep=";",
        encoding=linha_base["encoding"],
        low_memory=False
    )

    duplicados = dados[dados.duplicated(keep=False)].copy()

    exemplos_duplicados[dataset] = (
        duplicados
        .sort_values(list(dados.columns))
        .head(20)
    )

exemplos_duplicados["operacoes_indiretas_automaticas"]

## 34. Decisão preliminar sobre registros duplicados

Neste bloco, registramos a decisão metodológica preliminar para o tratamento dos duplicados.

A decisão considera a estrutura de cada base. Em operações não automáticas, existe número de contrato, o que permite maior segurança na identificação de duplicatas exatas. Em operações indiretas automáticas, a ausência de identificador único público exige mais cautela.

In [ ]:
decisao_duplicados = diagnostico_duplicados.copy()

def definir_tratamento_duplicado(dataset):
    if dataset == "operacoes_nao_automaticas":
        return "remover_duplicados_exatos"

    if dataset == "operacoes_indiretas_automaticas":
        return "criar_flag_e_avaliar_antes_de_remover"

    return "sem_duplicados_exatos"


decisao_duplicados["tratamento_preliminar"] = decisao_duplicados["dataset"].apply(
    definir_tratamento_duplicado
)

decisao_duplicados.sort_values(
    "percentual_duplicados",
    ascending=False
)

## 35. Diagnóstico de padronização textual

Neste bloco, verificamos problemas comuns em variáveis textuais, como espaços extras no início ou no fim dos textos.

Esse diagnóstico é importante porque diferenças aparentemente pequenas, como "SP" e " SP", podem gerar categorias duplicadas indevidas em tabelas, gráficos e agregações.

In [ ]:
diagnostico_texto = []

for _, linha in inventario_csv.iterrows():
    dataset = linha["dataset"]
    caminho = linha["caminho"]
    encoding = linha["encoding"]

    dados = pd.read_csv(
        caminho,
        sep=";",
        encoding=encoding,
        nrows=100000,
        low_memory=False
    )

    colunas_texto = dados.select_dtypes(include=["object", "string"]).columns

    for coluna in colunas_texto:
        serie = dados[coluna].dropna().astype("string")

        quantidade_com_espaco_extra = (
            serie.ne(serie.str.strip())
        ).sum()

        diagnostico_texto.append({
            "dataset": dataset,
            "coluna": coluna,
            "linhas_avaliadas": len(dados),
            "quantidade_com_espaco_extra": quantidade_com_espaco_extra,
            "percentual_com_espaco_extra": quantidade_com_espaco_extra / len(dados),
            "valores_unicos_amostra": serie.nunique()
        })

diagnostico_texto = pd.DataFrame(diagnostico_texto)

diagnostico_texto.sort_values(
    ["percentual_com_espaco_extra", "quantidade_com_espaco_extra"],
    ascending=False
)

## 36. Exemplos de padronização textual

Neste bloco, visualizamos exemplos de valores textuais antes e depois da remoção de espaços extras.

Essa inspeção permite confirmar que o tratamento proposto não altera o conteúdo da informação, apenas remove espaços no início ou no fim dos textos.

In [ ]:
colunas_com_espaco_extra = (
    diagnostico_texto
    .query("quantidade_com_espaco_extra > 0")
    .sort_values(
        ["percentual_com_espaco_extra", "quantidade_com_espaco_extra"],
        ascending=False
    )
    .head(15)
)

exemplos_padronizacao_texto = []

for _, linha in colunas_com_espaco_extra.iterrows():
    dataset = linha["dataset"]
    coluna = linha["coluna"]

    linha_base = inventario_csv.query("dataset == @dataset").iloc[0]

    dados = pd.read_csv(
        linha_base["caminho"],
        sep=";",
        encoding=linha_base["encoding"],
        usecols=[coluna],
        nrows=100000,
        low_memory=False
    )

    serie = dados[coluna].dropna().astype("string")
    valores_com_espaco = serie[serie.ne(serie.str.strip())].drop_duplicates().head(5)

    for valor in valores_com_espaco:
        exemplos_padronizacao_texto.append({
            "dataset": dataset,
            "coluna": coluna,
            "valor_original": f"[{valor}]",
            "valor_tratado": f"[{str(valor).strip()}]"
        })

exemplos_padronizacao_texto = pd.DataFrame(exemplos_padronizacao_texto)

exemplos_padronizacao_texto

## 37. Decisão de tratamento para variáveis textuais

Neste bloco, registramos a decisão de tratamento para as variáveis textuais.

A regra geral será remover espaços no início e no fim dos textos. Esse tratamento preserva o conteúdo original, mas evita categorias duplicadas indevidas em tabulações e agregações.

In [ ]:
decisao_texto = diagnostico_texto.copy()

decisao_texto["tratamento_preliminar"] = decisao_texto["quantidade_com_espaco_extra"].apply(
    lambda x: "aplicar_strip" if x > 0 else "sem_ajuste_textual"
)

decisao_texto_resumo = (
    decisao_texto
    .groupby(["dataset", "tratamento_preliminar"])
    .agg(
        quantidade_variaveis=("coluna", "count"),
        total_espacos_extras=("quantidade_com_espaco_extra", "sum")
    )
    .reset_index()
    .sort_values(["dataset", "tratamento_preliminar"])
)

decisao_texto_resumo

## 38. Lista de variáveis textuais a padronizar

Neste bloco, listamos as variáveis textuais que terão remoção de espaços extras no início e no fim dos textos.

Essa lista será usada na etapa de criação das bases tratadas.

In [ ]:
variaveis_texto_padronizar = (
    decisao_texto
    .query("tratamento_preliminar == 'aplicar_strip'")
    [["dataset", "coluna", "quantidade_com_espaco_extra", "percentual_com_espaco_extra"]]
    .sort_values(
        ["dataset", "percentual_com_espaco_extra", "quantidade_com_espaco_extra"],
        ascending=[True, False, False]
    )
    .reset_index(drop=True)
)

variaveis_texto_padronizar

## 39. Exportação dos diagnósticos de limpeza

Neste bloco, exportamos os principais diagnósticos de limpeza para um arquivo Excel.

Esse arquivo documenta as decisões sobre valores ausentes, duplicados e padronização textual antes da criação das bases tratadas.

Atenção: este bloco só deve ser executado quando os diagnósticos dos blocos 28 a 38 já tiverem sido rodados no kernel atual. Se o kernel foi reiniciado e o objetivo for continuar o tratamento, pule diretamente para o bloco 40.

In [ ]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
OUTPUT_TABLES_DIR = PROJECT_ROOT / "results" / "tables"
OUTPUT_TABLES_DIR.mkdir(parents=True, exist_ok=True)

objetos_necessarios = [
    "diagnostico_ausentes",
    "resumo_ausentes_dataset",
    "variaveis_com_ausentes",
    "decisao_ausentes",
    "diagnostico_duplicados",
    "decisao_duplicados",
    "diagnostico_texto",
    "variaveis_texto_padronizar",
]

objetos_ausentes = [
    nome for nome in objetos_necessarios
    if nome not in globals()
]

arquivo_diagnostico_limpeza = OUTPUT_TABLES_DIR / "diagnostico_limpeza_bases.xlsx"

if objetos_ausentes:
    print("Este bloco depende dos diagnósticos dos blocos 28 a 38.")
    print("Como esses objetos não estão carregados no kernel atual, a exportação não foi refeita.")
    print("Para continuar o tratamento sem recalcular os diagnósticos, pule para o bloco 40.")
    print("Objetos ausentes:")
    for nome in objetos_ausentes:
        print(f"- {nome}")
else:
    with pd.ExcelWriter(arquivo_diagnostico_limpeza, engine="openpyxl") as writer:
        diagnostico_ausentes.to_excel(
            writer,
            sheet_name="Ausentes_Completo",
            index=False,
        )

        resumo_ausentes_dataset.to_excel(
            writer,
            sheet_name="Ausentes_Resumo",
            index=False,
        )

        variaveis_com_ausentes.to_excel(
            writer,
            sheet_name="Variaveis_Ausentes",
            index=False,
        )

        decisao_ausentes.to_excel(
            writer,
            sheet_name="Decisao_Ausentes",
            index=False,
        )

        diagnostico_duplicados.to_excel(
            writer,
            sheet_name="Duplicados",
            index=False,
        )

        decisao_duplicados.to_excel(
            writer,
            sheet_name="Decisao_Duplicados",
            index=False,
        )

        diagnostico_texto.to_excel(
            writer,
            sheet_name="Texto_Completo",
            index=False,
        )

        variaveis_texto_padronizar.to_excel(
            writer,
            sheet_name="Texto_Padronizar",
            index=False,
        )

    print(f"Arquivo salvo: {arquivo_diagnostico_limpeza}")

arquivo_diagnostico_limpeza

## 40. Retomada rápida para continuar a limpeza

Neste bloco, recarregamos apenas os objetos essenciais para continuar o tratamento das bases sem executar novamente todos os diagnósticos pesados.

Ele também redefine as funções de limpeza. Assim, a seção final pode ser executada mesmo depois de reiniciar o kernel do Python.

In [ ]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

RAW_CSV_DIR = PROJECT_ROOT / "data" / "raw" / "csv"
INTERIM_DIR = PROJECT_ROOT / "data" / "interim"
OUTPUT_TABLES_DIR = PROJECT_ROOT / "results" / "tables"

DIAGNOSTICO_CSV_PATH = OUTPUT_TABLES_DIR / "diagnostico_leitura_csv.xlsx"

diagnostico_csv = pd.read_excel(DIAGNOSTICO_CSV_PATH)

mapa_nome_dataset = {
    "desembolsos_mensais__desembolsos_mensais.csv": "desembolsos_mensais",
    "desembolsos_mensais__mapeamento_de_bndes_para_cnae.csv": "mapeamento_bndes_cnae",
    "fontes_de_recursos__fontes_de_recursos_do_bndes.csv": "fontes_recursos",
    "operacoes_de_financiamento__operacoes_indiretas_automaticas.csv": "operacoes_indiretas_automaticas",
    "operacoes_de_financiamento__operacoes_nao_automaticas.csv": "operacoes_nao_automaticas",
    "politicas_operacionais__politicas_operacionais.csv": "politicas_operacionais",
}

inventario_csv = diagnostico_csv.copy()
inventario_csv["dataset"] = inventario_csv["nome_arquivo"].map(mapa_nome_dataset)
inventario_csv["caminho"] = inventario_csv["nome_arquivo"].apply(lambda nome: RAW_CSV_DIR / nome)

def limpar_textos(dados, colunas):
    dados = dados.copy()

    for coluna in colunas:
        if coluna in dados.columns:
            dados[coluna] = dados[coluna].astype("string").str.strip()

    return dados


def converter_decimal_brasileiro(serie):
    return (
        serie.astype("string")
        .str.strip()
        .str.replace(".", "", regex=False)
        .str.replace(",", ".", regex=False)
        .replace({"": pd.NA, "nan": pd.NA, "None": pd.NA})
        .pipe(pd.to_numeric, errors="coerce")
    )


def converter_percentual(serie):
    return (
        serie.astype("string")
        .str.strip()
        .str.replace("%", "", regex=False)
        .str.replace(",", ".", regex=False)
        .replace({"": pd.NA, "nan": pd.NA, "None": pd.NA})
        .pipe(pd.to_numeric, errors="coerce")
    )


def converter_data(serie):
    return pd.to_datetime(serie, errors="coerce")

inventario_csv[["dataset", "encoding", "caminho"]]

## 41. Tratamento da base fontes_recursos

Neste bloco, tratamos a base `fontes_recursos`.

Essa base é pequena e funciona como teste inicial do processo de limpeza. O tratamento inclui a conversão da coluna de data e a conversão das variáveis financeiras para formato numérico.

In [ ]:
linha_fontes = inventario_csv.loc[
    inventario_csv["dataset"].eq("fontes_recursos")
].iloc[0]

fontes_recursos_tratada = pd.read_csv(
    linha_fontes["caminho"],
    sep=";",
    encoding=linha_fontes["encoding"],
    low_memory=False,
)

fontes_recursos_tratada.columns = (
    fontes_recursos_tratada.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_", regex=False)
)

fontes_recursos_tratada["datas"] = converter_data(fontes_recursos_tratada["datas"])

colunas_financeiras_fontes = [
    "patrimonio_liquido",
    "tesouro_nacional",
    "fat",
    "captacoes_internas",
    "fundos",
    "operacoes_compromissadas",
    "captacoes_externas",
    "total_financeiro",
    "outros_passivos",
    "passivo_total",
]

for coluna in colunas_financeiras_fontes:
    fontes_recursos_tratada[coluna] = converter_decimal_brasileiro(
        fontes_recursos_tratada[coluna]
    ).astype("float64")

fontes_recursos_visualizacao = fontes_recursos_tratada.copy()
fontes_recursos_visualizacao["datas"] = fontes_recursos_visualizacao["datas"].dt.strftime("%d/%m/%Y")

fontes_recursos_visualizacao

## 42. Verificação dos tipos da base tratada

Neste bloco, conferimos se a data ficou em formato temporal e se as variáveis financeiras ficaram em formato numérico.

Essa checagem evita avançar com uma base que ainda esteja armazenando valores como texto.

In [ ]:
tipos_fontes_recursos = fontes_recursos_tratada.dtypes.reset_index()
tipos_fontes_recursos.columns = ["variavel", "tipo"]

tipos_fontes_recursos

## 43. Salvamento da base fontes_recursos tratada

Neste bloco, salvamos a base tratada no formato Parquet, dentro da pasta `data/interim`.

O Parquet preserva melhor os tipos das variáveis e é mais adequado para continuar o pipeline de tratamento e análise.

In [ ]:
INTERIM_DIR.mkdir(parents=True, exist_ok=True)

arquivo_fontes_tratada = INTERIM_DIR / "fontes_recursos.parquet"

fontes_recursos_tratada.to_parquet(
    arquivo_fontes_tratada,
    index=False,
    engine="pyarrow",
)

arquivo_fontes_tratada

## 44. Validação do arquivo salvo

Neste bloco, reabrimos o arquivo Parquet salvo para confirmar que ele foi gravado corretamente.

A validação mostra o número de linhas, o número de colunas e os tipos das variáveis depois da leitura.

In [ ]:
fontes_recursos_validacao = pd.read_parquet(
    arquivo_fontes_tratada,
    engine="pyarrow",
)

print(f"Linhas: {fontes_recursos_validacao.shape[0]}")
print(f"Colunas: {fontes_recursos_validacao.shape[1]}")

fontes_recursos_validacao_visualizacao = fontes_recursos_validacao.copy()
fontes_recursos_validacao_visualizacao["datas"] = fontes_recursos_validacao_visualizacao["datas"].dt.strftime("%d/%m/%Y")

fontes_recursos_validacao_visualizacao

## 45. Tratamento da base mapeamento_bndes_cnae

Neste bloco, tratamos a base `mapeamento_bndes_cnae`.

Essa base funciona como uma tabela auxiliar de classificação setorial. Ela relaciona setores e subsetores usados pelo BNDES com códigos CNAE do IBGE e será importante para organizar a análise setorial das operações e dos desembolsos.

Como a base não possui valores financeiros nem datas, o tratamento principal consiste em padronizar nomes das colunas, remover espaços extras em variáveis textuais e verificar duplicados exatos.

In [ ]:
linha_mapeamento = inventario_csv.loc[
    inventario_csv["dataset"].eq("mapeamento_bndes_cnae")
].iloc[0]

mapeamento_bndes_cnae_tratado = pd.read_csv(
    linha_mapeamento["caminho"],
    sep=";",
    encoding=linha_mapeamento["encoding"],
    low_memory=False,
)

mapeamento_bndes_cnae_tratado.columns = (
    mapeamento_bndes_cnae_tratado.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_", regex=False)
)

colunas_texto_mapeamento = mapeamento_bndes_cnae_tratado.select_dtypes(
    include=["object", "string"]
).columns

mapeamento_bndes_cnae_tratado = limpar_textos(
    mapeamento_bndes_cnae_tratado,
    colunas_texto_mapeamento
)

linhas_antes = len(mapeamento_bndes_cnae_tratado)

mapeamento_bndes_cnae_tratado = (
    mapeamento_bndes_cnae_tratado
    .drop_duplicates()
    .reset_index(drop=True)
)

linhas_depois = len(mapeamento_bndes_cnae_tratado)

print(f"Linhas antes: {linhas_antes}")
print(f"Linhas depois: {linhas_depois}")
print(f"Duplicados removidos: {linhas_antes - linhas_depois}")

mapeamento_bndes_cnae_tratado

## 46. Salvamento da base mapeamento_bndes_cnae tratada

Neste bloco, salvamos a base `mapeamento_bndes_cnae` tratada em formato Parquet.

Essa base será usada como tabela auxiliar para análise setorial, permitindo relacionar os setores e subsetores do BNDES aos códigos CNAE do IBGE.

In [ ]:
arquivo_mapeamento_tratado = INTERIM_DIR / "mapeamento_bndes_cnae.parquet"

mapeamento_bndes_cnae_tratado.to_parquet(
    arquivo_mapeamento_tratado,
    index=False,
    engine="pyarrow",
)

arquivo_mapeamento_tratado

## 47. Validação da base mapeamento_bndes_cnae salva

Neste bloco, reabrimos o arquivo Parquet da base `mapeamento_bndes_cnae` para confirmar que ele foi salvo corretamente.

A validação mostra o número de linhas, o número de colunas e uma prévia da base tratada.

In [ ]:
mapeamento_bndes_cnae_validacao = pd.read_parquet(
    arquivo_mapeamento_tratado,
    engine="pyarrow",
)

print(f"Linhas: {mapeamento_bndes_cnae_validacao.shape[0]}")
print(f"Colunas: {mapeamento_bndes_cnae_validacao.shape[1]}")

mapeamento_bndes_cnae_validacao

## 48. Tratamento da base politicas_operacionais

Neste bloco, tratamos a base `politicas_operacionais`.

Essa base descreve modalidades, tipos de apoio, instrumentos, linhas, sublinhas, custos financeiros, taxas, prazos, participação máxima do BNDES e garantias. Ela será importante para interpretar as condições das linhas de financiamento e identificar instrumentos com possível finalidade ambiental.

O tratamento inclui padronização textual, conversão das taxas percentuais para formato numérico e criação de uma variável auxiliar de prazo em meses.

In [ ]:
linha_politicas = inventario_csv.loc[
    inventario_csv["dataset"].eq("politicas_operacionais")
].iloc[0]

politicas_operacionais_tratada = pd.read_csv(
    linha_politicas["caminho"],
    sep=";",
    encoding=linha_politicas["encoding"],
    low_memory=False,
)

politicas_operacionais_tratada.columns = (
    politicas_operacionais_tratada.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_", regex=False)
)

colunas_texto_politicas = politicas_operacionais_tratada.select_dtypes(
    include=["object", "string"]
).columns

politicas_operacionais_tratada = limpar_textos(
    politicas_operacionais_tratada,
    colunas_texto_politicas
)

politicas_operacionais_tratada = politicas_operacionais_tratada.replace(
    {"-": pd.NA}
)

politicas_operacionais_tratada["taxa_bndes_a_a_numero"] = converter_percentual(
    politicas_operacionais_tratada["taxa_bndes_a_a"]
)

politicas_operacionais_tratada["participacao_maxima_bndes_numero"] = converter_percentual(
    politicas_operacionais_tratada["participacao_maxima_bndes"]
)

def converter_prazo_para_meses(serie):
    texto = serie.astype("string").str.lower().str.strip()

    anos = texto.str.extract(r"(\d+(?:[,.]\d+)?)\s*ano")[0]
    meses = texto.str.extract(r"(\d+(?:[,.]\d+)?)\s*m[eê]s")[0]

    anos = pd.to_numeric(
        anos.str.replace(",", ".", regex=False),
        errors="coerce"
    ).fillna(0)

    meses = pd.to_numeric(
        meses.str.replace(",", ".", regex=False),
        errors="coerce"
    ).fillna(0)

    prazo_meses = (anos * 12) + meses
    prazo_meses = prazo_meses.where(texto.notna(), pd.NA)

    return prazo_meses.astype("Float64")

politicas_operacionais_tratada["prazo_total_maximo_meses"] = converter_prazo_para_meses(
    politicas_operacionais_tratada["prazo_total_maximo"]
)

print(f"Linhas: {politicas_operacionais_tratada.shape[0]}")
print(f"Colunas: {politicas_operacionais_tratada.shape[1]}")

politicas_operacionais_tratada

## 49. Validação das conversões da base politicas_operacionais

Neste bloco, verificamos se as conversões realizadas na base `politicas_operacionais` ficaram coerentes.

A checagem compara as variáveis originais com as novas variáveis numéricas. O objetivo é identificar casos em que havia informação textual, mas a conversão para número não foi possível. Esses casos precisam ser documentados antes de salvar a base tratada.

In [ ]:
mapa_conversoes_politicas = {
    "taxa_bndes_a_a": "taxa_bndes_a_a_numero",
    "participacao_maxima_bndes": "participacao_maxima_bndes_numero",
    "prazo_total_maximo": "prazo_total_maximo_meses",
}

diagnostico_conversoes_politicas = []

for coluna_original, coluna_convertida in mapa_conversoes_politicas.items():
    original_preenchido = politicas_operacionais_tratada[coluna_original].notna()
    convertido_ausente = politicas_operacionais_tratada[coluna_convertida].isna()

    diagnostico_conversoes_politicas.append({
        "variavel_original": coluna_original,
        "variavel_convertida": coluna_convertida,
        "linhas_total": len(politicas_operacionais_tratada),
        "original_preenchido": original_preenchido.sum(),
        "convertido_ausente": convertido_ausente.sum(),
        "original_preenchido_mas_nao_convertido": (
            original_preenchido & convertido_ausente
        ).sum(),
    })

diagnostico_conversoes_politicas = pd.DataFrame(
    diagnostico_conversoes_politicas
)

diagnostico_conversoes_politicas

## 50. Registros com conversão pendente em politicas_operacionais

Neste bloco, listamos os registros em que alguma variável original estava preenchida, mas a variável numérica correspondente ficou ausente.

Essa inspeção ajuda a diferenciar ausência real de informação, como fundos não reembolsáveis, de casos que exigem regra adicional de tratamento.

In [ ]:
mascara_conversao_pendente = False

for coluna_original, coluna_convertida in mapa_conversoes_politicas.items():
    mascara_conversao_pendente = (
        mascara_conversao_pendente
        | (
            politicas_operacionais_tratada[coluna_original].notna()
            & politicas_operacionais_tratada[coluna_convertida].isna()
        )
    )

politicas_conversao_pendente = politicas_operacionais_tratada.loc[
    mascara_conversao_pendente,
    [
        "modalidade",
        "tipo_de_apoio",
        "instrumento_de_apoio_po",
        "linha_po",
        "taxa_bndes_a_a",
        "taxa_bndes_a_a_numero",
        "participacao_maxima_bndes",
        "participacao_maxima_bndes_numero",
        "prazo_total_maximo",
        "prazo_total_maximo_meses",
    ]
].reset_index(drop=True)

politicas_conversao_pendente

## 51. Ajuste das conversões percentuais em politicas_operacionais

Neste bloco, refinamos a conversão das variáveis percentuais da base `politicas_operacionais`.

Algumas células trazem percentuais dentro de textos, como `100% do valor da exportação` ou `80% do valor do compromisso de exportação`. Nesses casos, extraímos o percentual inicial para criar uma variável numérica.

Por outro lado, expressões como `Condições Específicas`, `Conforme Finem`, `Não Reembolsável` e `Não se aplica` são mantidas como ausência na variável numérica, pois não representam um valor percentual diretamente observável.

In [ ]:
def extrair_percentual_do_texto(serie):
    texto = serie.astype("string").str.strip()

    percentual = texto.str.extract(
        r"(\d+(?:[,.]\d+)?)\s*%"
    )[0]

    percentual = (
        percentual
        .str.replace(",", ".", regex=False)
        .pipe(pd.to_numeric, errors="coerce")
    )

    return percentual.astype("Float64")


politicas_operacionais_tratada["taxa_bndes_a_a_numero"] = extrair_percentual_do_texto(
    politicas_operacionais_tratada["taxa_bndes_a_a"]
)

politicas_operacionais_tratada["participacao_maxima_bndes_numero"] = extrair_percentual_do_texto(
    politicas_operacionais_tratada["participacao_maxima_bndes"]
)

diagnostico_conversoes_politicas_ajustado = []

for coluna_original, coluna_convertida in mapa_conversoes_politicas.items():
    original_preenchido = politicas_operacionais_tratada[coluna_original].notna()
    convertido_ausente = politicas_operacionais_tratada[coluna_convertida].isna()

    diagnostico_conversoes_politicas_ajustado.append({
        "variavel_original": coluna_original,
        "variavel_convertida": coluna_convertida,
        "linhas_total": len(politicas_operacionais_tratada),
        "original_preenchido": original_preenchido.sum(),
        "convertido_ausente": convertido_ausente.sum(),
        "original_preenchido_mas_nao_convertido": (
            original_preenchido & convertido_ausente
        ).sum(),
    })

diagnostico_conversoes_politicas_ajustado = pd.DataFrame(
    diagnostico_conversoes_politicas_ajustado
)

diagnostico_conversoes_politicas_ajustado

## 52. Valores ainda não convertidos em politicas_operacionais

Neste bloco, verificamos quais valores textuais ainda não foram convertidos para número após o ajuste das variáveis percentuais.

Essa etapa serve para decidir se esses casos devem receber uma nova regra de conversão ou se devem permanecer como ausentes na variável numérica, por representarem condições específicas, não aplicáveis ou não reembolsáveis.

In [ ]:
mapa_conversoes_politicas = {
    "taxa_bndes_a_a": "taxa_bndes_a_a_numero",
    "participacao_maxima_bndes": "participacao_maxima_bndes_numero",
    "prazo_total_maximo": "prazo_total_maximo_meses",
}

valores_nao_convertidos = []

for coluna_original, coluna_convertida in mapa_conversoes_politicas.items():
    mascara = (
        politicas_operacionais_tratada[coluna_original].notna()
        & politicas_operacionais_tratada[coluna_convertida].isna()
    )

    resumo_valores = (
        politicas_operacionais_tratada.loc[mascara, coluna_original]
        .value_counts(dropna=False)
        .reset_index()
    )

    resumo_valores.columns = ["valor_original", "quantidade"]
    resumo_valores["variavel_original"] = coluna_original
    resumo_valores["variavel_convertida"] = coluna_convertida

    valores_nao_convertidos.append(resumo_valores)

valores_nao_convertidos_politicas = (
    pd.concat(valores_nao_convertidos, ignore_index=True)
    .loc[
        :,
        [
            "variavel_original",
            "variavel_convertida",
            "valor_original",
            "quantidade",
        ],
    ]
    .sort_values(
        ["variavel_original", "quantidade"],
        ascending=[True, False],
    )
    .reset_index(drop=True)
)

valores_nao_convertidos_politicas

## 53. Classificação das conversões pendentes em politicas_operacionais

Neste bloco, classificamos os valores que permaneceram sem conversão numérica.

Esses casos não representam erro de tratamento. Em geral, indicam condições específicas, instrumentos não reembolsáveis, regras condicionais ou ausência real de informação. Por isso, eles serão mantidos como texto nas variáveis originais e como ausentes nas variáveis numéricas.

In [ ]:
classificacao_valores_nao_convertidos = {
    "Condições Específicas": "condicao_especifica",
    "Não Reembolsável": "nao_reembolsavel",
    "Não se aplica": "nao_se_aplica",
    "Conforme Finem": "conforme_linha_finem",
    "1/6 dos bens": "regra_especifica_nao_percentual",
    "nan%": "ausencia_real",
}

valores_nao_convertidos_politicas["classificacao_tratamento"] = (
    valores_nao_convertidos_politicas["valor_original"]
    .map(classificacao_valores_nao_convertidos)
    .fillna("verificar")
)

valores_nao_convertidos_politicas

## 54. Salvamento da base politicas_operacionais tratada

Neste bloco, salvamos a base `politicas_operacionais` tratada em formato Parquet.

Também salvamos a tabela de valores não convertidos, pois ela documenta quais casos permaneceram como texto por representarem condições específicas, regras não percentuais, instrumentos não reembolsáveis ou ausência real de informação.

In [ ]:
arquivo_politicas_tratada = INTERIM_DIR / "politicas_operacionais.parquet"
arquivo_politicas_conversoes = OUTPUT_TABLES_DIR / "politicas_operacionais_conversoes_pendentes.xlsx"

politicas_operacionais_tratada.to_parquet(
    arquivo_politicas_tratada,
    index=False,
    engine="pyarrow",
)

valores_nao_convertidos_politicas.to_excel(
    arquivo_politicas_conversoes,
    index=False,
    sheet_name="Conversoes_Pendentes",
)

arquivo_politicas_tratada

## 55. Validação da base politicas_operacionais salva

Neste bloco, reabrimos a base `politicas_operacionais` salva em Parquet para confirmar que o arquivo foi gravado corretamente.

A validação mostra o número de linhas, o número de colunas e uma prévia da base tratada.

In [ ]:
politicas_operacionais_validacao = pd.read_parquet(
    arquivo_politicas_tratada,
    engine="pyarrow",
)

print(f"Linhas: {politicas_operacionais_validacao.shape[0]}")
print(f"Colunas: {politicas_operacionais_validacao.shape[1]}")

politicas_operacionais_validacao

## 56. Tratamento da base operacoes_nao_automaticas

Neste bloco, iniciamos o tratamento da base `operacoes_nao_automaticas`.

Essa base contém operações de financiamento que passaram por análise não automática. Ela é importante para o projeto porque traz informações detalhadas sobre cliente, CNPJ, descrição do projeto, UF, município, setor, subsetor, produto, fonte de recurso, valor contratado e valor desembolsado.

Como essa base é menor que a de operações indiretas automáticas, ela será usada como teste para o tratamento das bases operacionais.

In [ ]:
linha_operacoes_nao_auto = inventario_csv.loc[
    inventario_csv["dataset"].eq("operacoes_nao_automaticas")
].iloc[0]

operacoes_nao_automaticas_tratada = pd.read_csv(
    linha_operacoes_nao_auto["caminho"],
    sep=";",
    encoding=linha_operacoes_nao_auto["encoding"],
    low_memory=False,
)

operacoes_nao_automaticas_tratada.columns = (
    operacoes_nao_automaticas_tratada.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_", regex=False)
)

colunas_texto_operacoes_nao_auto = operacoes_nao_automaticas_tratada.select_dtypes(
    include=["object", "string"]
).columns

operacoes_nao_automaticas_tratada = limpar_textos(
    operacoes_nao_automaticas_tratada,
    colunas_texto_operacoes_nao_auto
)

operacoes_nao_automaticas_tratada = operacoes_nao_automaticas_tratada.replace(
    {"": pd.NA, "-": pd.NA}
)

print(f"Linhas: {operacoes_nao_automaticas_tratada.shape[0]}")
print(f"Colunas: {operacoes_nao_automaticas_tratada.shape[1]}")

operacoes_nao_automaticas_tratada

## 57. Diagnóstico dos tipos da base operacoes_nao_automaticas

Neste bloco, verificamos os tipos das variáveis da base `operacoes_nao_automaticas` após a padronização textual inicial.

A base contém variáveis de identificação, data, localização, setor, instrumento financeiro e valores monetários. Antes de salvar a base tratada, precisamos confirmar quais colunas devem permanecer como texto e quais devem ser convertidas para data ou número.

In [ ]:
diagnostico_tipos_operacoes_nao_auto = pd.DataFrame({
    "coluna": operacoes_nao_automaticas_tratada.columns,
    "tipo_atual": [
        operacoes_nao_automaticas_tratada[coluna].dtype
        for coluna in operacoes_nao_automaticas_tratada.columns
    ],
    "quantidade_ausentes": [
        operacoes_nao_automaticas_tratada[coluna].isna().sum()
        for coluna in operacoes_nao_automaticas_tratada.columns
    ],
    "percentual_ausente": [
        operacoes_nao_automaticas_tratada[coluna].isna().mean()
        for coluna in operacoes_nao_automaticas_tratada.columns
    ],
    "exemplo_1": [
        operacoes_nao_automaticas_tratada[coluna].dropna().iloc[0]
        if operacoes_nao_automaticas_tratada[coluna].notna().any()
        else pd.NA
        for coluna in operacoes_nao_automaticas_tratada.columns
    ],
})

diagnostico_tipos_operacoes_nao_auto

## 58. Conversão de tipos da base operacoes_nao_automaticas

Neste bloco, corrigimos os tipos das principais variáveis da base `operacoes_nao_automaticas`.

A data de contratação será convertida para formato temporal, os valores monetários e juros serão convertidos para formato numérico, e os códigos identificadores serão mantidos como texto. Essa decisão evita tratar códigos de município e números de contrato como variáveis quantitativas.

In [ ]:
operacoes_nao_automaticas_tratada["data_da_contratacao"] = converter_data(
    operacoes_nao_automaticas_tratada["data_da_contratacao"]
)

colunas_valores_operacoes_nao_auto = [
    "valor_contratado_reais",
    "valor_desembolsado_reais",
]

for coluna in colunas_valores_operacoes_nao_auto:
    operacoes_nao_automaticas_tratada[coluna] = converter_decimal_brasileiro(
        operacoes_nao_automaticas_tratada[coluna]
    ).astype("float64")

operacoes_nao_automaticas_tratada["juros_numero"] = converter_decimal_brasileiro(
    operacoes_nao_automaticas_tratada["juros"]
).astype("float64")

operacoes_nao_automaticas_tratada["municipio_codigo"] = (
    operacoes_nao_automaticas_tratada["municipio_codigo"]
    .astype("string")
    .str.zfill(7)
)

operacoes_nao_automaticas_tratada["numero_do_contrato"] = (
    operacoes_nao_automaticas_tratada["numero_do_contrato"]
    .astype("string")
)

operacoes_nao_automaticas_visualizacao = operacoes_nao_automaticas_tratada.copy()
operacoes_nao_automaticas_visualizacao["data_da_contratacao"] = (
    operacoes_nao_automaticas_visualizacao["data_da_contratacao"]
    .dt.strftime("%d/%m/%Y")
)

operacoes_nao_automaticas_visualizacao

## 59. Validação das conversões em operacoes_nao_automaticas

Neste bloco, validamos se as conversões de tipo da base `operacoes_nao_automaticas` foram aplicadas corretamente.

A verificação confirma se a data de contratação está em formato temporal, se os valores monetários e juros estão numéricos, e se os identificadores foram mantidos como texto.

In [ ]:
colunas_validacao_operacoes_nao_auto = [
    "municipio_codigo",
    "numero_do_contrato",
    "data_da_contratacao",
    "valor_contratado_reais",
    "valor_desembolsado_reais",
    "juros",
    "juros_numero",
]

validacao_tipos_operacoes_nao_auto = pd.DataFrame({
    "coluna": colunas_validacao_operacoes_nao_auto,
    "tipo": [
        operacoes_nao_automaticas_tratada[coluna].dtype
        for coluna in colunas_validacao_operacoes_nao_auto
    ],
    "quantidade_ausentes": [
        operacoes_nao_automaticas_tratada[coluna].isna().sum()
        for coluna in colunas_validacao_operacoes_nao_auto
    ],
    "exemplo_1": [
        operacoes_nao_automaticas_tratada[coluna].dropna().iloc[0]
        if operacoes_nao_automaticas_tratada[coluna].notna().any()
        else pd.NA
        for coluna in colunas_validacao_operacoes_nao_auto
    ],
    "exemplo_2": [
        operacoes_nao_automaticas_tratada[coluna].dropna().iloc[-1]
        if operacoes_nao_automaticas_tratada[coluna].notna().any()
        else pd.NA
        for coluna in colunas_validacao_operacoes_nao_auto
    ],
})

validacao_tipos_operacoes_nao_auto

## 60. Verificação de duplicados em operacoes_nao_automaticas

Neste bloco, verificamos a presença de registros duplicados na base `operacoes_nao_automaticas`.

O diagnóstico anterior indicou a existência de duplicados exatos nessa base. Como estamos preparando uma base tratada para análise, precisamos quantificar esses registros e removê-los apenas quando forem duplicações idênticas em todas as colunas.

In [ ]:
linhas_antes_duplicados_nao_auto = len(operacoes_nao_automaticas_tratada)

duplicados_exatos_nao_auto = operacoes_nao_automaticas_tratada.duplicated(
    keep=False
)

resumo_duplicados_nao_auto = pd.DataFrame([{
    "linhas_antes": linhas_antes_duplicados_nao_auto,
    "registros_em_grupos_duplicados": duplicados_exatos_nao_auto.sum(),
    "duplicados_excedentes": operacoes_nao_automaticas_tratada.duplicated().sum(),
    "percentual_duplicados_excedentes": (
        operacoes_nao_automaticas_tratada.duplicated().mean()
    ),
}])

resumo_duplicados_nao_auto

## 61. Remoção de duplicados exatos em operacoes_nao_automaticas

Neste bloco, removemos os duplicados exatos da base `operacoes_nao_automaticas`.

A remoção considera todas as colunas da base tratada. Assim, apenas registros integralmente idênticos são deduplicados, preservando operações que possam ter o mesmo contrato, cliente ou município, mas diferenças em valores, fontes, produtos ou demais variáveis.

In [ ]:
operacoes_nao_automaticas_tratada = (
    operacoes_nao_automaticas_tratada
    .drop_duplicates()
    .reset_index(drop=True)
)

resumo_pos_duplicados_nao_auto = pd.DataFrame([{
    "linhas_antes": linhas_antes_duplicados_nao_auto,
    "linhas_depois": len(operacoes_nao_automaticas_tratada),
    "duplicados_removidos": (
        linhas_antes_duplicados_nao_auto
        - len(operacoes_nao_automaticas_tratada)
    ),
    "duplicados_restantes": operacoes_nao_automaticas_tratada.duplicated().sum(),
}])

resumo_pos_duplicados_nao_auto

## 62. Salvamento da base operacoes_nao_automaticas tratada

Neste bloco, salvamos a base `operacoes_nao_automaticas` tratada em formato Parquet.

A base salva já contém a padronização textual, conversão de data, conversão de valores monetários, conversão de juros para variável numérica, códigos identificadores como texto e remoção de duplicados exatos.

In [ ]:
arquivo_operacoes_nao_auto_tratada = (
    INTERIM_DIR / "operacoes_nao_automaticas.parquet"
)

operacoes_nao_automaticas_tratada.to_parquet(
    arquivo_operacoes_nao_auto_tratada,
    index=False,
    engine="pyarrow",
)

arquivo_operacoes_nao_auto_tratada

## 63. Validação da base operacoes_nao_automaticas salva

Neste bloco, reabrimos a base `operacoes_nao_automaticas` salva em Parquet para confirmar que o arquivo foi gravado corretamente.

A validação mostra o número de linhas, o número de colunas e uma prévia da base tratada.

In [ ]:
operacoes_nao_automaticas_validacao = pd.read_parquet(
    arquivo_operacoes_nao_auto_tratada,
    engine="pyarrow",
)

print(f"Linhas: {operacoes_nao_automaticas_validacao.shape[0]}")
print(f"Colunas: {operacoes_nao_automaticas_validacao.shape[1]}")

operacoes_nao_automaticas_validacao_visualizacao = (
    operacoes_nao_automaticas_validacao.copy()
)

operacoes_nao_automaticas_validacao_visualizacao["data_da_contratacao"] = (
    operacoes_nao_automaticas_validacao_visualizacao["data_da_contratacao"]
    .dt.strftime("%d/%m/%Y")
)

operacoes_nao_automaticas_validacao_visualizacao

## 64. Amostra inicial da base desembolsos_mensais

Neste bloco, carregamos uma amostra da base `desembolsos_mensais`.

Essa base é grande, com mais de 3 milhões de linhas. Por isso, antes de aplicar qualquer tratamento no arquivo completo, vamos testar as regras em uma amostra de 100 mil linhas. Essa estratégia reduz o risco de erro e evita travamentos no notebook.

In [ ]:
linha_desembolsos = inventario_csv.loc[
    inventario_csv["dataset"].eq("desembolsos_mensais")
].iloc[0]

desembolsos_mensais_amostra = pd.read_csv(
    linha_desembolsos["caminho"],
    sep=";",
    encoding=linha_desembolsos["encoding"],
    nrows=100_000,
    low_memory=False,
)

desembolsos_mensais_amostra.columns = (
    desembolsos_mensais_amostra.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_", regex=False)
)

colunas_texto_desembolsos_amostra = desembolsos_mensais_amostra.select_dtypes(
    include=["object", "string"]
).columns

desembolsos_mensais_amostra = limpar_textos(
    desembolsos_mensais_amostra,
    colunas_texto_desembolsos_amostra,
)

desembolsos_mensais_amostra = desembolsos_mensais_amostra.replace(
    {"": pd.NA, "-": pd.NA}
)

print(f"Linhas da amostra: {desembolsos_mensais_amostra.shape[0]}")
print(f"Colunas: {desembolsos_mensais_amostra.shape[1]}")

desembolsos_mensais_amostra

## 65. Diagnóstico dos tipos da amostra de desembolsos_mensais

Neste bloco, verificamos os tipos das variáveis da amostra de `desembolsos_mensais`.

Antes de tratar a base completa, precisamos identificar quais colunas devem ser convertidas para data, número ou texto identificador.

In [ ]:
diagnostico_tipos_desembolsos_amostra = pd.DataFrame({
    "coluna": desembolsos_mensais_amostra.columns,
    "tipo_atual": [
        desembolsos_mensais_amostra[coluna].dtype
        for coluna in desembolsos_mensais_amostra.columns
    ],
    "quantidade_ausentes": [
        desembolsos_mensais_amostra[coluna].isna().sum()
        for coluna in desembolsos_mensais_amostra.columns
    ],
    "percentual_ausente": [
        desembolsos_mensais_amostra[coluna].isna().mean()
        for coluna in desembolsos_mensais_amostra.columns
    ],
    "exemplo_1": [
        desembolsos_mensais_amostra[coluna].dropna().iloc[0]
        if desembolsos_mensais_amostra[coluna].notna().any()
        else pd.NA
        for coluna in desembolsos_mensais_amostra.columns
    ],
})

diagnostico_tipos_desembolsos_amostra

## 66. Conversão de tipos na amostra de desembolsos_mensais

Neste bloco, aplicamos as principais conversões de tipo na amostra de `desembolsos_mensais`.

A base possui ano e mês separados, que serão combinados em uma variável temporal `ano_mes`. O valor de desembolso será convertido para número e o código do município será mantido como texto identificador.

In [ ]:
desembolsos_mensais_amostra_tratada = desembolsos_mensais_amostra.copy()

desembolsos_mensais_amostra_tratada["ano"] = pd.to_numeric(
    desembolsos_mensais_amostra_tratada["ano"],
    errors="coerce",
).astype("Int64")

desembolsos_mensais_amostra_tratada["mes"] = pd.to_numeric(
    desembolsos_mensais_amostra_tratada["mes"],
    errors="coerce",
).astype("Int64")

desembolsos_mensais_amostra_tratada["ano_mes"] = pd.to_datetime(
    dict(
        year=desembolsos_mensais_amostra_tratada["ano"],
        month=desembolsos_mensais_amostra_tratada["mes"],
        day=1,
    ),
    errors="coerce",
)

desembolsos_mensais_amostra_tratada["municipio_codigo"] = (
    desembolsos_mensais_amostra_tratada["municipio_codigo"]
    .astype("string")
    .str.zfill(7)
)

desembolsos_mensais_amostra_tratada["desembolsos_reais"] = converter_decimal_brasileiro(
    desembolsos_mensais_amostra_tratada["desembolsos_reais"]
).astype("float64")

desembolsos_mensais_amostra_visualizacao = desembolsos_mensais_amostra_tratada.copy()
desembolsos_mensais_amostra_visualizacao["ano_mes"] = (
    desembolsos_mensais_amostra_visualizacao["ano_mes"]
    .dt.strftime("%d/%m/%Y")
)

desembolsos_mensais_amostra_visualizacao

## 67. Validação das conversões na amostra de desembolsos_mensais

Neste bloco, validamos as conversões aplicadas na amostra de `desembolsos_mensais`.

A checagem confirma se `ano`, `mes`, `ano_mes`, `municipio_codigo` e `desembolsos_reais` ficaram nos tipos esperados antes de aplicar o tratamento ao arquivo completo.

In [ ]:
colunas_validacao_desembolsos_amostra = [
    "ano",
    "mes",
    "ano_mes",
    "municipio_codigo",
    "desembolsos_reais",
]

validacao_tipos_desembolsos_amostra = pd.DataFrame({
    "coluna": colunas_validacao_desembolsos_amostra,
    "tipo": [
        desembolsos_mensais_amostra_tratada[coluna].dtype
        for coluna in colunas_validacao_desembolsos_amostra
    ],
    "quantidade_ausentes": [
        desembolsos_mensais_amostra_tratada[coluna].isna().sum()
        for coluna in colunas_validacao_desembolsos_amostra
    ],
    "exemplo_1": [
        desembolsos_mensais_amostra_tratada[coluna].dropna().iloc[0]
        if desembolsos_mensais_amostra_tratada[coluna].notna().any()
        else pd.NA
        for coluna in colunas_validacao_desembolsos_amostra
    ],
    "exemplo_2": [
        desembolsos_mensais_amostra_tratada[coluna].dropna().iloc[-1]
        if desembolsos_mensais_amostra_tratada[coluna].notna().any()
        else pd.NA
        for coluna in colunas_validacao_desembolsos_amostra
    ],
})

validacao_tipos_desembolsos_amostra

## 68. Tratamento completo da base desembolsos_mensais

Neste bloco, aplicamos o tratamento validado na amostra ao arquivo completo de `desembolsos_mensais`.

Como a base possui mais de 3 milhões de linhas, o processamento será feito em partes (`chunks`). Cada parte é limpa, convertida e gravada no mesmo arquivo Parquet. Essa abordagem evita carregar toda a base na memória do computador.

In [ ]:
import pyarrow as pa
import pyarrow.parquet as pq

arquivo_desembolsos_tratado = INTERIM_DIR / "desembolsos_mensais.parquet"
tamanho_chunk_desembolsos = 250_000

if arquivo_desembolsos_tratado.exists():
    arquivo_desembolsos_tratado.unlink()

def tratar_chunk_desembolsos(chunk):
    chunk = chunk.copy()

    chunk.columns = (
        chunk.columns
        .str.strip()
        .str.lower()
        .str.replace(" ", "_", regex=False)
    )

    colunas_texto = chunk.select_dtypes(include=["object", "string"]).columns
    chunk = limpar_textos(chunk, colunas_texto)
    chunk = chunk.replace({"": pd.NA, "-": pd.NA})

    chunk["ano"] = pd.to_numeric(chunk["ano"], errors="coerce").astype("Int64")
    chunk["mes"] = pd.to_numeric(chunk["mes"], errors="coerce").astype("Int64")

    chunk["ano_mes"] = pd.to_datetime(
        dict(
            year=chunk["ano"],
            month=chunk["mes"],
            day=1,
        ),
        errors="coerce",
    )

    chunk["municipio_codigo"] = (
        chunk["municipio_codigo"]
        .astype("string")
        .str.zfill(7)
    )

    chunk["desembolsos_reais"] = converter_decimal_brasileiro(
        chunk["desembolsos_reais"]
    ).astype("float64")

    colunas_ordenadas = [
        "ano",
        "mes",
        "ano_mes",
        "forma_de_apoio",
        "produto",
        "instrumento_financeiro",
        "inovacao",
        "porte_de_empresa",
        "regiao",
        "uf",
        "municipio",
        "municipio_codigo",
        "setor_cnae",
        "subsetor_cnae_agrupado",
        "setor_bndes",
        "subsetor_bndes",
        "desembolsos_reais",
    ]

    return chunk[colunas_ordenadas]


resumo_chunks_desembolsos = []
parquet_writer = None

leitor_chunks_desembolsos = pd.read_csv(
    linha_desembolsos["caminho"],
    sep=";",
    encoding=linha_desembolsos["encoding"],
    chunksize=tamanho_chunk_desembolsos,
    low_memory=False,
)

for numero_chunk, chunk in enumerate(leitor_chunks_desembolsos, start=1):
    chunk_tratado = tratar_chunk_desembolsos(chunk)
    tabela_arrow = pa.Table.from_pandas(chunk_tratado, preserve_index=False)

    if parquet_writer is None:
        parquet_writer = pq.ParquetWriter(
            arquivo_desembolsos_tratado,
            tabela_arrow.schema,
            compression="snappy",
        )

    parquet_writer.write_table(tabela_arrow)

    resumo_chunks_desembolsos.append({
        "chunk": numero_chunk,
        "linhas": len(chunk_tratado),
        "primeiro_ano_mes": chunk_tratado["ano_mes"].min(),
        "ultimo_ano_mes": chunk_tratado["ano_mes"].max(),
        "valor_desembolsado_total": chunk_tratado["desembolsos_reais"].sum(),
    })

if parquet_writer is not None:
    parquet_writer.close()

resumo_tratamento_desembolsos_mensais = pd.DataFrame(resumo_chunks_desembolsos)
resumo_tratamento_desembolsos_mensais

## 69. Validação do arquivo desembolsos_mensais salvo

Neste bloco, validamos o arquivo Parquet da base `desembolsos_mensais` sem carregar a base completa no notebook.

A validação usa os metadados do Parquet para conferir número de linhas, número de colunas, tamanho do arquivo e uma pequena amostra inicial para visualização no Data Wrangler.

In [ ]:
parquet_desembolsos = pq.ParquetFile(arquivo_desembolsos_tratado)

resumo_validacao_desembolsos_mensais = pd.DataFrame([{
    "arquivo": arquivo_desembolsos_tratado.name,
    "linhas": parquet_desembolsos.metadata.num_rows,
    "colunas": parquet_desembolsos.metadata.num_columns,
    "grupos_linhas": parquet_desembolsos.metadata.num_row_groups,
    "tamanho_mb": arquivo_desembolsos_tratado.stat().st_size / 1024**2,
}])

amostra_validacao_desembolsos = (
    parquet_desembolsos
    .read_row_group(0)
    .slice(0, 50)
    .to_pandas()
)

amostra_validacao_desembolsos_visualizacao = amostra_validacao_desembolsos.copy()
amostra_validacao_desembolsos_visualizacao["ano_mes"] = (
    amostra_validacao_desembolsos_visualizacao["ano_mes"]
    .dt.strftime("%d/%m/%Y")
)

resumo_validacao_desembolsos_mensais

## 70. Amostra de validação de desembolsos_mensais

Neste bloco, visualizamos uma pequena amostra do arquivo Parquet já salvo.

Essa saída deve ser aberta no Data Wrangler para verificar se as variáveis aparecem corretamente, especialmente `ano_mes`, `municipio_codigo` e `desembolsos_reais`.

In [ ]:
amostra_validacao_desembolsos_visualizacao

## 71. Validação dos tipos em desembolsos_mensais salvo

Neste bloco, conferimos os tipos das principais variáveis na amostra lida diretamente do arquivo Parquet salvo.

Essa verificação confirma se o tratamento aplicado em chunks preservou os tipos esperados no arquivo final.

In [ ]:
colunas_validacao_desembolsos = [
    "ano",
    "mes",
    "ano_mes",
    "municipio_codigo",
    "desembolsos_reais",
]

validacao_tipos_desembolsos_mensais = pd.DataFrame({
    "coluna": colunas_validacao_desembolsos,
    "tipo": [
        amostra_validacao_desembolsos[coluna].dtype
        for coluna in colunas_validacao_desembolsos
    ],
    "quantidade_ausentes_amostra": [
        amostra_validacao_desembolsos[coluna].isna().sum()
        for coluna in colunas_validacao_desembolsos
    ],
    "exemplo_1": [
        amostra_validacao_desembolsos[coluna].dropna().iloc[0]
        if amostra_validacao_desembolsos[coluna].notna().any()
        else pd.NA
        for coluna in colunas_validacao_desembolsos
    ],
})

validacao_tipos_desembolsos_mensais

## 72. Amostra inicial da base operacoes_indiretas_automaticas

Neste bloco, carregamos uma amostra da base `operacoes_indiretas_automaticas`.

Essa é a maior base de operações do projeto, com mais de 2 milhões de linhas. Antes de aplicar o tratamento completo, vamos testar as regras em uma amostra de 100 mil linhas.

In [ ]:
linha_operacoes_indiretas = inventario_csv.loc[
    inventario_csv["dataset"].eq("operacoes_indiretas_automaticas")
].iloc[0]

operacoes_indiretas_amostra = pd.read_csv(
    linha_operacoes_indiretas["caminho"],
    sep=";",
    encoding=linha_operacoes_indiretas["encoding"],
    nrows=100_000,
    low_memory=False,
)

operacoes_indiretas_amostra.columns = (
    operacoes_indiretas_amostra.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_", regex=False)
)

colunas_texto_operacoes_indiretas_amostra = operacoes_indiretas_amostra.select_dtypes(
    include=["object", "string"]
).columns

operacoes_indiretas_amostra = limpar_textos(
    operacoes_indiretas_amostra,
    colunas_texto_operacoes_indiretas_amostra,
)

operacoes_indiretas_amostra = operacoes_indiretas_amostra.replace(
    {"": pd.NA, "-": pd.NA}
)

print(f"Linhas da amostra: {operacoes_indiretas_amostra.shape[0]}")
print(f"Colunas: {operacoes_indiretas_amostra.shape[1]}")

operacoes_indiretas_amostra

## 73. Diagnóstico dos tipos da amostra de operacoes_indiretas_automaticas

Neste bloco, verificamos os tipos das variáveis da amostra de `operacoes_indiretas_automaticas`.

O objetivo é identificar quais colunas precisam ser convertidas para data, número ou texto identificador antes de processar a base completa.

In [ ]:
diagnostico_tipos_operacoes_indiretas_amostra = pd.DataFrame({
    "coluna": operacoes_indiretas_amostra.columns,
    "tipo_atual": [
        operacoes_indiretas_amostra[coluna].dtype
        for coluna in operacoes_indiretas_amostra.columns
    ],
    "quantidade_ausentes": [
        operacoes_indiretas_amostra[coluna].isna().sum()
        for coluna in operacoes_indiretas_amostra.columns
    ],
    "percentual_ausente": [
        operacoes_indiretas_amostra[coluna].isna().mean()
        for coluna in operacoes_indiretas_amostra.columns
    ],
    "exemplo_1": [
        operacoes_indiretas_amostra[coluna].dropna().iloc[0]
        if operacoes_indiretas_amostra[coluna].notna().any()
        else pd.NA
        for coluna in operacoes_indiretas_amostra.columns
    ],
})

diagnostico_tipos_operacoes_indiretas_amostra

## 74. Conversão de tipos na amostra de operacoes_indiretas_automaticas

Neste bloco, aplicamos as conversões principais na amostra de `operacoes_indiretas_automaticas`.

A data de contratação será convertida para formato temporal, os valores monetários e juros serão convertidos para formato numérico, e os códigos identificadores serão mantidos como texto.

In [ ]:
operacoes_indiretas_amostra_tratada = operacoes_indiretas_amostra.copy()

operacoes_indiretas_amostra_tratada["data_da_contratacao"] = converter_data(
    operacoes_indiretas_amostra_tratada["data_da_contratacao"]
)

colunas_valores_operacoes_indiretas = [
    "valor_da_operacao_em_reais",
    "valor_desembolsado_reais",
]

for coluna in colunas_valores_operacoes_indiretas:
    operacoes_indiretas_amostra_tratada[coluna] = converter_decimal_brasileiro(
        operacoes_indiretas_amostra_tratada[coluna]
    ).astype("float64")

operacoes_indiretas_amostra_tratada["juros_numero"] = converter_decimal_brasileiro(
    operacoes_indiretas_amostra_tratada["juros"]
).astype("float64")

operacoes_indiretas_amostra_tratada["municipio_codigo"] = (
    operacoes_indiretas_amostra_tratada["municipio_codigo"]
    .astype("string")
    .str.zfill(7)
)

operacoes_indiretas_amostra_tratada["cpf_cnpj"] = (
    operacoes_indiretas_amostra_tratada["cpf_cnpj"]
    .astype("string")
)

operacoes_indiretas_visualizacao = operacoes_indiretas_amostra_tratada.copy()
operacoes_indiretas_visualizacao["data_da_contratacao"] = (
    operacoes_indiretas_visualizacao["data_da_contratacao"]
    .dt.strftime("%d/%m/%Y")
)

operacoes_indiretas_visualizacao

## 75. Validação das conversões na amostra de operacoes_indiretas_automaticas

Neste bloco, validamos se as conversões de tipo da amostra de `operacoes_indiretas_automaticas` foram aplicadas corretamente.

A checagem confirma se a data está em formato temporal, se valores monetários e juros estão numéricos, e se identificadores foram mantidos como texto.

In [ ]:
colunas_validacao_operacoes_indiretas = [
    "cpf_cnpj",
    "municipio_codigo",
    "data_da_contratacao",
    "valor_da_operacao_em_reais",
    "valor_desembolsado_reais",
    "juros",
    "juros_numero",
]

validacao_tipos_operacoes_indiretas_amostra = pd.DataFrame({
    "coluna": colunas_validacao_operacoes_indiretas,
    "tipo": [
        operacoes_indiretas_amostra_tratada[coluna].dtype
        for coluna in colunas_validacao_operacoes_indiretas
    ],
    "quantidade_ausentes": [
        operacoes_indiretas_amostra_tratada[coluna].isna().sum()
        for coluna in colunas_validacao_operacoes_indiretas
    ],
    "exemplo_1": [
        operacoes_indiretas_amostra_tratada[coluna].dropna().iloc[0]
        if operacoes_indiretas_amostra_tratada[coluna].notna().any()
        else pd.NA
        for coluna in colunas_validacao_operacoes_indiretas
    ],
    "exemplo_2": [
        operacoes_indiretas_amostra_tratada[coluna].dropna().iloc[-1]
        if operacoes_indiretas_amostra_tratada[coluna].notna().any()
        else pd.NA
        for coluna in colunas_validacao_operacoes_indiretas
    ],
})

validacao_tipos_operacoes_indiretas_amostra

## 76. Tratamento completo da base operacoes_indiretas_automaticas

Neste bloco, aplicamos o tratamento validado na amostra ao arquivo completo de `operacoes_indiretas_automaticas`.

Como essa base possui mais de 2 milhões de linhas, o processamento será feito em partes (`chunks`). Nesta etapa, ainda não removemos duplicados, pois a proporção de duplicação nessa base é maior e precisa ser analisada separadamente.

In [ ]:
import pyarrow as pa
import pyarrow.parquet as pq

arquivo_operacoes_indiretas_tratada = (
    INTERIM_DIR / "operacoes_indiretas_automaticas.parquet"
)
tamanho_chunk_operacoes_indiretas = 200_000

if arquivo_operacoes_indiretas_tratada.exists():
    arquivo_operacoes_indiretas_tratada.unlink()

def tratar_chunk_operacoes_indiretas(chunk):
    chunk = chunk.copy()

    chunk.columns = (
        chunk.columns
        .str.strip()
        .str.lower()
        .str.replace(" ", "_", regex=False)
    )

    colunas_texto = chunk.select_dtypes(include=["object", "string"]).columns
    chunk = limpar_textos(chunk, colunas_texto)
    chunk = chunk.replace({"": pd.NA, "-": pd.NA})

    chunk["data_da_contratacao"] = converter_data(chunk["data_da_contratacao"])

    colunas_valores = [
        "valor_da_operacao_em_reais",
        "valor_desembolsado_reais",
    ]

    for coluna in colunas_valores:
        chunk[coluna] = converter_decimal_brasileiro(
            chunk[coluna]
        ).astype("float64")

    chunk["juros_numero"] = converter_decimal_brasileiro(
        chunk["juros"]
    ).astype("float64")

    chunk["municipio_codigo"] = (
        chunk["municipio_codigo"]
        .astype("string")
        .str.zfill(7)
    )

    chunk["cpf_cnpj"] = chunk["cpf_cnpj"].astype("string")
    chunk["cnpj_do_agente_financeiro"] = (
        chunk["cnpj_do_agente_financeiro"]
        .astype("string")
    )

    for coluna in ["prazo_carencia_meses", "prazo_amortizacao_meses"]:
        chunk[coluna] = pd.to_numeric(chunk[coluna], errors="coerce").astype("Int64")

    colunas_ordenadas = [
        "cliente",
        "cpf_cnpj",
        "uf",
        "municipio",
        "municipio_codigo",
        "data_da_contratacao",
        "valor_da_operacao_em_reais",
        "valor_desembolsado_reais",
        "fonte_de_recurso_desembolsos",
        "custo_financeiro",
        "juros",
        "juros_numero",
        "prazo_carencia_meses",
        "prazo_amortizacao_meses",
        "modalidade_de_apoio",
        "forma_de_apoio",
        "produto",
        "instrumento_financeiro",
        "inovacao",
        "area_operacional",
        "setor_cnae",
        "subsetor_cnae_agrupado",
        "subsetor_cnae_codigo",
        "subsetor_cnae_nome",
        "setor_bndes",
        "subsetor_bndes",
        "porte_do_cliente",
        "natureza_do_cliente",
        "instituicao_financeira_credenciada",
        "cnpj_do_agente_financeiro",
        "situacao_da_operacao",
    ]

    return chunk[colunas_ordenadas]


resumo_chunks_operacoes_indiretas = []
parquet_writer = None

leitor_chunks_operacoes_indiretas = pd.read_csv(
    linha_operacoes_indiretas["caminho"],
    sep=";",
    encoding=linha_operacoes_indiretas["encoding"],
    chunksize=tamanho_chunk_operacoes_indiretas,
    low_memory=False,
)

for numero_chunk, chunk in enumerate(leitor_chunks_operacoes_indiretas, start=1):
    chunk_tratado = tratar_chunk_operacoes_indiretas(chunk)
    tabela_arrow = pa.Table.from_pandas(chunk_tratado, preserve_index=False)

    if parquet_writer is None:
        parquet_writer = pq.ParquetWriter(
            arquivo_operacoes_indiretas_tratada,
            tabela_arrow.schema,
            compression="snappy",
        )

    parquet_writer.write_table(tabela_arrow)

    resumo_chunks_operacoes_indiretas.append({
        "chunk": numero_chunk,
        "linhas": len(chunk_tratado),
        "primeira_data": chunk_tratado["data_da_contratacao"].min(),
        "ultima_data": chunk_tratado["data_da_contratacao"].max(),
        "valor_operacao_total": chunk_tratado["valor_da_operacao_em_reais"].sum(),
        "valor_desembolsado_total": chunk_tratado["valor_desembolsado_reais"].sum(),
    })

if parquet_writer is not None:
    parquet_writer.close()

resumo_tratamento_operacoes_indiretas = pd.DataFrame(
    resumo_chunks_operacoes_indiretas
)

resumo_tratamento_operacoes_indiretas

## 77. Validação do arquivo operacoes_indiretas_automaticas salvo

Neste bloco, validamos o arquivo Parquet da base `operacoes_indiretas_automaticas` sem carregar a base completa no notebook.

A validação usa metadados do Parquet para conferir número de linhas, número de colunas, número de grupos de linhas e tamanho do arquivo.

In [ ]:
parquet_operacoes_indiretas = pq.ParquetFile(
    arquivo_operacoes_indiretas_tratada
)

resumo_validacao_operacoes_indiretas = pd.DataFrame([{
    "arquivo": arquivo_operacoes_indiretas_tratada.name,
    "linhas": parquet_operacoes_indiretas.metadata.num_rows,
    "colunas": parquet_operacoes_indiretas.metadata.num_columns,
    "grupos_linhas": parquet_operacoes_indiretas.metadata.num_row_groups,
    "tamanho_mb": arquivo_operacoes_indiretas_tratada.stat().st_size / 1024**2,
}])

amostra_validacao_operacoes_indiretas = (
    parquet_operacoes_indiretas
    .read_row_group(0)
    .slice(0, 50)
    .to_pandas()
)

amostra_validacao_operacoes_indiretas_visualizacao = (
    amostra_validacao_operacoes_indiretas.copy()
)
amostra_validacao_operacoes_indiretas_visualizacao["data_da_contratacao"] = (
    amostra_validacao_operacoes_indiretas_visualizacao["data_da_contratacao"]
    .dt.strftime("%d/%m/%Y")
)

resumo_validacao_operacoes_indiretas

## 78. Amostra de validação de operacoes_indiretas_automaticas

Neste bloco, visualizamos uma pequena amostra da base `operacoes_indiretas_automaticas` já salva em Parquet.

A saída deve ser aberta no Data Wrangler para verificar se as principais variáveis aparecem corretamente após o tratamento.

In [ ]:
amostra_validacao_operacoes_indiretas_visualizacao

## 79. Validação dos tipos em operacoes_indiretas_automaticas salva

Neste bloco, conferimos os tipos das principais variáveis na amostra lida diretamente do arquivo Parquet salvo.

Essa verificação confirma se o tratamento aplicado em chunks preservou os tipos esperados no arquivo final.

In [ ]:
colunas_validacao_operacoes_indiretas_final = [
    "cpf_cnpj",
    "municipio_codigo",
    "data_da_contratacao",
    "valor_da_operacao_em_reais",
    "valor_desembolsado_reais",
    "juros",
    "juros_numero",
    "prazo_carencia_meses",
    "prazo_amortizacao_meses",
]

validacao_tipos_operacoes_indiretas_final = pd.DataFrame({
    "coluna": colunas_validacao_operacoes_indiretas_final,
    "tipo": [
        amostra_validacao_operacoes_indiretas[coluna].dtype
        for coluna in colunas_validacao_operacoes_indiretas_final
    ],
    "quantidade_ausentes_amostra": [
        amostra_validacao_operacoes_indiretas[coluna].isna().sum()
        for coluna in colunas_validacao_operacoes_indiretas_final
    ],
    "exemplo_1": [
        amostra_validacao_operacoes_indiretas[coluna].dropna().iloc[0]
        if amostra_validacao_operacoes_indiretas[coluna].notna().any()
        else pd.NA
        for coluna in colunas_validacao_operacoes_indiretas_final
    ],
})

validacao_tipos_operacoes_indiretas_final

## 80. Diagnóstico de duplicados em operacoes_indiretas_automaticas

Neste bloco, verificamos duplicados na base `operacoes_indiretas_automaticas` salva em Parquet.

Como a base é grande, o diagnóstico usa um hash calculado a partir de todas as colunas de cada linha. Esse procedimento permite identificar linhas integralmente repetidas sem abrir toda a base no Data Wrangler.

In [ ]:
parquet_operacoes_indiretas = pq.ParquetFile(
    arquivo_operacoes_indiretas_tratada
)

contagens_hash_operacoes_indiretas = []

for grupo_linha in range(parquet_operacoes_indiretas.metadata.num_row_groups):
    chunk = parquet_operacoes_indiretas.read_row_group(grupo_linha).to_pandas()

    hash_linha = pd.util.hash_pandas_object(
        chunk.astype("string").fillna("<NA>"),
        index=False,
    )

    contagens_hash_operacoes_indiretas.append(
        hash_linha.value_counts()
    )

contagem_hash_operacoes_indiretas = (
    pd.concat(contagens_hash_operacoes_indiretas)
    .groupby(level=0)
    .sum()
)

hashes_duplicados_operacoes_indiretas = (
    contagem_hash_operacoes_indiretas[contagem_hash_operacoes_indiretas > 1]
    .sort_values(ascending=False)
)

resumo_duplicados_operacoes_indiretas = pd.DataFrame([{
    "linhas_total": int(parquet_operacoes_indiretas.metadata.num_rows),
    "grupos_duplicados": int(len(hashes_duplicados_operacoes_indiretas)),
    "registros_em_grupos_duplicados": int(hashes_duplicados_operacoes_indiretas.sum()),
    "duplicados_excedentes": int((hashes_duplicados_operacoes_indiretas - 1).sum()),
    "percentual_duplicados_excedentes": float(
        (hashes_duplicados_operacoes_indiretas - 1).sum()
        / parquet_operacoes_indiretas.metadata.num_rows
    ),
}])

resumo_duplicados_operacoes_indiretas

## 81. Inspeção de exemplos duplicados em operacoes_indiretas_automaticas

Neste bloco, inspecionamos alguns grupos de linhas duplicadas da base `operacoes_indiretas_automaticas`.

A ideia é verificar se os grupos identificados pelo hash correspondem, de fato, a registros repetidos integralmente. Por enquanto, esses registros não serão removidos; a decisão será tomada após a inspeção.

In [ ]:
hashes_para_inspecao_indiretas = list(
    hashes_duplicados_operacoes_indiretas.head(10).index
)

amostras_duplicados_indiretas = []

for grupo_linha in range(parquet_operacoes_indiretas.metadata.num_row_groups):
    chunk = parquet_operacoes_indiretas.read_row_group(grupo_linha).to_pandas()

    hash_linha = pd.util.hash_pandas_object(
        chunk.astype("string").fillna("<NA>"),
        index=False,
    )

    mascara = hash_linha.isin(hashes_para_inspecao_indiretas)
    if mascara.any():
        chunk_amostra = chunk.loc[mascara].copy()
        chunk_amostra["hash_linha"] = hash_linha.loc[mascara].astype("string").values
        amostras_duplicados_indiretas.append(chunk_amostra)

amostra_duplicados_indiretas = pd.concat(
    amostras_duplicados_indiretas,
    ignore_index=True,
)

amostra_duplicados_indiretas_visualizacao = amostra_duplicados_indiretas.copy()
amostra_duplicados_indiretas_visualizacao["data_da_contratacao"] = (
    amostra_duplicados_indiretas_visualizacao["data_da_contratacao"]
    .dt.strftime("%d/%m/%Y")
)

colunas_inspecao_duplicados_indiretas = [
    "hash_linha",
    "cliente",
    "cpf_cnpj",
    "uf",
    "municipio",
    "municipio_codigo",
    "data_da_contratacao",
    "valor_da_operacao_em_reais",
    "valor_desembolsado_reais",
    "produto",
    "instrumento_financeiro",
    "situacao_da_operacao",
]

amostra_duplicados_indiretas_visualizacao[
    colunas_inspecao_duplicados_indiretas
].sort_values(["hash_linha", "cliente"]).reset_index(drop=True)

## 82. Documentação do diagnóstico de duplicados em operacoes_indiretas_automaticas

Neste bloco, salvamos o resumo de duplicados da base `operacoes_indiretas_automaticas` em Excel.

Essa documentação é importante porque a base possui um volume relevante de registros repetidos. Diferentemente da base `operacoes_nao_automaticas`, aqui a decisão de remover duplicados será feita somente depois de validar os exemplos e registrar a justificativa.

In [ ]:
arquivo_duplicados_operacoes_indiretas = (
    OUTPUT_TABLES_DIR / "operacoes_indiretas_duplicados.xlsx"
)

top_hashes_duplicados_operacoes_indiretas = (
    hashes_duplicados_operacoes_indiretas
    .head(100)
    .reset_index()
)
top_hashes_duplicados_operacoes_indiretas.columns = [
    "hash_linha",
    "quantidade_registros",
]

with pd.ExcelWriter(arquivo_duplicados_operacoes_indiretas, engine="openpyxl") as writer:
    resumo_duplicados_operacoes_indiretas.to_excel(
        writer,
        sheet_name="Resumo",
        index=False,
    )
    top_hashes_duplicados_operacoes_indiretas.to_excel(
        writer,
        sheet_name="Top_Hashes",
        index=False,
    )

arquivo_duplicados_operacoes_indiretas

## 83. Inventário final das bases tratadas

Neste bloco, consolidamos as bases já salvas na pasta `data/interim`.

O objetivo é fechar a etapa de limpeza com uma tabela de controle contendo nome do arquivo, número de linhas, número de colunas, tamanho e data de modificação. Essa tabela será útil para a próxima etapa de análise descritiva e exploratória.

In [ ]:
arquivos_parquet_interim = sorted(INTERIM_DIR.glob("*.parquet"))

inventario_bases_tratadas = []

for arquivo in arquivos_parquet_interim:
    parquet_file = pq.ParquetFile(arquivo)

    inventario_bases_tratadas.append({
        "base_tratada": arquivo.stem,
        "arquivo": arquivo.name,
        "linhas": parquet_file.metadata.num_rows,
        "colunas": parquet_file.metadata.num_columns,
        "grupos_linhas": parquet_file.metadata.num_row_groups,
        "tamanho_mb": arquivo.stat().st_size / 1024**2,
        "ultima_modificacao": pd.to_datetime(
            arquivo.stat().st_mtime,
            unit="s",
        ),
    })

inventario_bases_tratadas = pd.DataFrame(inventario_bases_tratadas)

inventario_bases_tratadas_visualizacao = inventario_bases_tratadas.copy()
inventario_bases_tratadas_visualizacao["ultima_modificacao"] = (
    inventario_bases_tratadas_visualizacao["ultima_modificacao"]
    .dt.strftime("%d/%m/%Y %H:%M")
)

inventario_bases_tratadas_visualizacao

## 84. Registro das principais decisões de tratamento

Neste bloco, registramos as principais decisões metodológicas tomadas durante a limpeza.

Esse registro é importante para garantir reprodutibilidade e transparência. Ele documenta quais bases tiveram conversões de tipos, remoção de duplicados, tratamento de ausentes ou apenas padronização textual.

In [ ]:
decisoes_tratamento_bases = pd.DataFrame([
    {
        "base": "fontes_recursos",
        "tratamento_principal": "Conversão de datas e variáveis financeiras.",
        "duplicados": "Não identificados.",
        "observacao": "Base pequena; mantida como tabela auxiliar de fontes e funding.",
    },
    {
        "base": "mapeamento_bndes_cnae",
        "tratamento_principal": "Padronização textual e remoção de espaços extras.",
        "duplicados": "Não identificados.",
        "observacao": "Tabela auxiliar para compatibilização setorial BNDES-CNAE.",
    },
    {
        "base": "politicas_operacionais",
        "tratamento_principal": "Conversão de percentuais e prazo máximo para meses.",
        "duplicados": "Não identificados na etapa final.",
        "observacao": "Valores não convertidos documentados em Excel por representarem condições específicas, não aplicáveis ou não reembolsáveis.",
    },
    {
        "base": "operacoes_nao_automaticas",
        "tratamento_principal": "Conversão de data, valores monetários, juros e identificadores.",
        "duplicados": "379 duplicados exatos removidos.",
        "observacao": "Base salva já deduplicada, com 23.104 linhas.",
    },
    {
        "base": "desembolsos_mensais",
        "tratamento_principal": "Criação de ano_mes, conversão de desembolsos para número e município como identificador textual.",
        "duplicados": "Não removidos nesta etapa.",
        "observacao": "Processada em chunks por ser base grande, com 3.762.192 linhas.",
    },
    {
        "base": "operacoes_indiretas_automaticas",
        "tratamento_principal": "Conversão de data, valores monetários, juros, prazos e identificadores.",
        "duplicados": "193.530 duplicados excedentes identificados, mas não removidos.",
        "observacao": "Duplicados documentados separadamente; remoção adiada para etapa analítica específica.",
    },
])

decisoes_tratamento_bases

## 85. Exportação do resumo final da limpeza

Neste bloco, exportamos o inventário final das bases tratadas e o registro das decisões de tratamento para um arquivo Excel.

Esse arquivo serve como documentação da etapa de limpeza e pode ser compartilhado ou consultado antes de iniciar a análise descritiva.

In [ ]:
arquivo_resumo_final_limpeza = (
    OUTPUT_TABLES_DIR / "resumo_final_limpeza_bases.xlsx"
)

with pd.ExcelWriter(arquivo_resumo_final_limpeza, engine="openpyxl") as writer:
    inventario_bases_tratadas_visualizacao.to_excel(
        writer,
        sheet_name="Bases_Tratadas",
        index=False,
    )
    decisoes_tratamento_bases.to_excel(
        writer,
        sheet_name="Decisoes_Tratamento",
        index=False,
    )

arquivo_resumo_final_limpeza

## 86. Publicação das bases finais em data/processed

Neste bloco, copiamos as bases tratadas da pasta `data/interim` para a pasta `data/processed`.

A pasta `data/interim` representa a etapa intermediária de tratamento. Já `data/processed` armazena as versões finais, limpas e prontas para análise ou compartilhamento.

In [ ]:
from shutil import copy2
import pyarrow.parquet as pq

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

arquivos_publicados_processed = []

for arquivo_origem in sorted(INTERIM_DIR.glob("*.parquet")):
    arquivo_destino = PROCESSED_DIR / arquivo_origem.name
    copy2(arquivo_origem, arquivo_destino)

    parquet_file = pq.ParquetFile(arquivo_destino)

    arquivos_publicados_processed.append({
        "base_processada": arquivo_destino.stem,
        "arquivo": arquivo_destino.name,
        "linhas": parquet_file.metadata.num_rows,
        "colunas": parquet_file.metadata.num_columns,
        "grupos_linhas": parquet_file.metadata.num_row_groups,
        "tamanho_mb": arquivo_destino.stat().st_size / 1024**2,
        "pasta": str(PROCESSED_DIR),
    })

inventario_bases_processadas = pd.DataFrame(arquivos_publicados_processed)
inventario_bases_processadas

## 87. Exportação do resumo final das bases processadas

Neste bloco, exportamos o inventário das bases finais publicadas em `data/processed`.

Esse arquivo Excel é a documentação mais adequada para acompanhar as bases que serão compartilhadas ou usadas na etapa de análise.

In [ ]:
arquivo_resumo_bases_processadas = (
    OUTPUT_TABLES_DIR / "resumo_bases_processadas.xlsx"
)

with pd.ExcelWriter(arquivo_resumo_bases_processadas, engine="openpyxl") as writer:
    inventario_bases_processadas.to_excel(
        writer,
        sheet_name="Bases_Processadas",
        index=False,
    )
    decisoes_tratamento_bases.to_excel(
        writer,
        sheet_name="Decisoes_Tratamento",
        index=False,
    )

arquivo_resumo_bases_processadas

## 88. Encerramento da etapa de limpeza

A etapa de limpeza e tratamento das bases foi concluída.

As versões finais para análise e compartilhamento estão salvas em `data/processed` no formato Parquet. A próxima etapa do projeto será a análise descritiva e exploratória.

In [ ]:
print("Etapa de limpeza concluída.")
print(f"Bases processadas: {len(inventario_bases_processadas)}")
print(f"Pasta final: {PROCESSED_DIR}")
print(f"Resumo exportado: {arquivo_resumo_bases_processadas}")